In [1]:
# ============================================================
# PHASE 3A — EXPANDED 47-PREDICTOR REGISTRY + DATA DISCOVERY
# ============================================================
#
# PURPOSE
# -------
# 1. Lock the intended expanded predictor universe.
# 2. Confirm that the intended count is exactly 47 conceptual
#    predictors.
# 3. Treat READ / MATH / SCIENCE achievement correctly as
#    matched 10-plausible-value families.
# 4. Locate the frozen per-system processed datasets used by
#    the earlier analysis.
# 5. Inspect schemas ONLY.
#
# NO MODEL FITTING.
# NO FEATURE SELECTION.
# NO NEW MISSINGNESS RULES YET.
# ============================================================


from pathlib import Path
import re
import gc

import numpy as np
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path("../results")

OUT_DIR = (
    RESULT_ROOT
    / "expanded_predictor_analysis"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RANKING_PATH = (
    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv"
)


# ============================================================
# 2. Frozen 14-predictor core
# ============================================================

CORE14 = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


# ============================================================
# 3. Intended expanded NON-ACHIEVEMENT predictors
#
# 44 ordinary constructs.
#
# NOTE:
# The old source list contained a missing-comma bug near
# PA006Q09TA in one version. We define the intended registry
# explicitly here instead of trusting that old Python list.
# ============================================================

EXPANDED_NONACHIEVEMENT = [

    # --------------------------------------------------------
    # INDIVIDUAL
    # --------------------------------------------------------

    "ST004D01T",     # Gender
    "AGE",
    "GRADE",
    "BSMJ",          # Expected occupational status

    "JOYREAD",
    "SCREADCOMP",
    "SCREADDIFF",

    "COMPETE",
    "WORKMAST",
    "GFOFAIL",
    "EUDMO",
    "RESILIENCE",
    "MASTGOAL",

    "ST185Q01HA",    # Meaning / purpose in life item
    "ST184Q01HA",    # Growth mindset item

    "SWBP",          # Subjective well-being

    # Bullying-attitude items
    "ST207Q01HA",
    "ST207Q02HA",
    "ST207Q03HA",
    "ST207Q04HA",
    "ST207Q05HA",

    # --------------------------------------------------------
    # PROXIMAL
    # --------------------------------------------------------

    "REPEAT",
    "UNDREM",
    "METASUM",
    "METASPAM",

    # --------------------------------------------------------
    # MICROSYSTEM / SCHOOL-EXPERIENCE
    # --------------------------------------------------------

    "EMOSUPS",
    "DURECEC",
    "BELONG",
    "PERCOMP",
    "PERCOOP",

    "ATTLNACT",
    "DISCLIMA",
    "TEACHSUP",
    "DIRINS",
    "PERFEED",
    "STIMREAD",
    "ADAPTIVITY",
    "TEACHINT",

    "ST206Q01HA",

    # Parent questionnaire item.
    # Expected to be a structural availability bottleneck.
    "PA006Q09TA",

    # --------------------------------------------------------
    # MACRO / EXOSYSTEM-RELATED STUDENT/SCHOOL CONTEXT
    # --------------------------------------------------------

    "ESCS",
    "EDUSHORT",
    "RATCMP1",
    "RATCMP2",
]


# ============================================================
# 4. Achievement families
#
# These are THREE conceptual predictors.
#
# During actual modeling:
#
# replicate i uses:
#
#   PViREAD
#   PViMATH
#   PViSCIE
#
# for i = 1,...,10.
#
# We therefore NEVER use PV1 only.
# ============================================================

ACHIEVEMENT_FAMILIES = {

    "READ": [
        f"PV{i}READ"
        for i in range(1, 11)
    ],

    "MATH": [
        f"PV{i}MATH"
        for i in range(1, 11)
    ],

    "SCIE": [
        f"PV{i}SCIE"
        for i in range(1, 11)
    ],
}


ACHIEVEMENT_CONCEPTS = [
    "PV_READ",
    "PV_MATH",
    "PV_SCIE",
]


# ============================================================
# 5. Sanity checks on the intended predictor universe
# ============================================================

if len(
    EXPANDED_NONACHIEVEMENT
) != 44:

    raise RuntimeError(
        "Expanded non-achievement list should contain "
        f"44 predictors, found {len(EXPANDED_NONACHIEVEMENT)}."
    )


if len(
    set(
        EXPANDED_NONACHIEVEMENT
    )
) != 44:

    duplicates = (
        pd.Series(
            EXPANDED_NONACHIEVEMENT
        )
        .value_counts()
    )

    duplicates = duplicates[
        duplicates > 1
    ]

    raise RuntimeError(
        f"Duplicate predictors found:\n{duplicates}"
    )


if not set(
    CORE14
).issubset(
    set(
        EXPANDED_NONACHIEVEMENT
    )
):

    missing_core = sorted(
        set(
            CORE14
        )
        -
        set(
            EXPANDED_NONACHIEVEMENT
        )
    )

    raise RuntimeError(
        "Expanded list is missing frozen core predictors: "
        f"{missing_core}"
    )


N_CONCEPTUAL = (
    len(
        EXPANDED_NONACHIEVEMENT
    )
    +
    len(
        ACHIEVEMENT_CONCEPTS
    )
)


assert N_CONCEPTUAL == 47


print(
    "Expanded non-achievement predictors:",
    len(
        EXPANDED_NONACHIEVEMENT
    )
)

print(
    "Achievement concepts:",
    len(
        ACHIEVEMENT_CONCEPTS
    )
)

print(
    "TOTAL conceptual predictors:",
    N_CONCEPTUAL
)


# ============================================================
# 6. Predictor registry
# ============================================================

LEVEL_MAP = {}


# Individual
for code in [
    "ST004D01T",
    "AGE",
    "GRADE",
    "BSMJ",
    "JOYREAD",
    "SCREADCOMP",
    "SCREADDIFF",
    "COMPETE",
    "WORKMAST",
    "GFOFAIL",
    "EUDMO",
    "RESILIENCE",
    "MASTGOAL",
    "ST185Q01HA",
    "ST184Q01HA",
    "SWBP",
    "ST207Q01HA",
    "ST207Q02HA",
    "ST207Q03HA",
    "ST207Q04HA",
    "ST207Q05HA",
]:

    LEVEL_MAP[
        code
    ] = "Individual"


# Proximal
for code in [
    "REPEAT",
    "UNDREM",
    "METASUM",
    "METASPAM",
]:

    LEVEL_MAP[
        code
    ] = "Proximal"


# Microsystem
for code in [
    "EMOSUPS",
    "DURECEC",
    "BELONG",
    "PERCOMP",
    "PERCOOP",
    "ATTLNACT",
    "DISCLIMA",
    "TEACHSUP",
    "DIRINS",
    "PERFEED",
    "STIMREAD",
    "ADAPTIVITY",
    "TEACHINT",
    "ST206Q01HA",
    "PA006Q09TA",
]:

    LEVEL_MAP[
        code
    ] = "Microsystem"


# Macro / exosystem
for code in [
    "ESCS",
    "EDUSHORT",
    "RATCMP1",
    "RATCMP2",
]:

    LEVEL_MAP[
        code
    ] = "Macro/Exosystem"


CONSTRUCT_LABELS = {

    "ST004D01T":
        "Gender",

    "AGE":
        "Age",

    "GRADE":
        "Grade",

    "BSMJ":
        "Expected occupational status",

    "JOYREAD":
        "Joy of reading",

    "SCREADCOMP":
        "Reading self-concept: competence",

    "SCREADDIFF":
        "Reading self-concept: difficulty",

    "COMPETE":
        "Competitiveness",

    "WORKMAST":
        "Work mastery orientation",

    "GFOFAIL":
        "General fear of failure",

    "EUDMO":
        "Sense of meaning in life / eudaimonia",

    "RESILIENCE":
        "Resilience",

    "MASTGOAL":
        "Mastery goal orientation",

    "ST185Q01HA":
        "Meaning/purpose item",

    "ST184Q01HA":
        "Growth mindset item",

    "SWBP":
        "Subjective well-being",

    "REPEAT":
        "Grade repetition",

    "UNDREM":
        "Undermining / related index",

    "METASUM":
        "Metacognition summary",

    "METASPAM":
        "Metacognition / spam-related reading strategy index",

    "EMOSUPS":
        "Emotional support",

    "DURECEC":
        "Duration of early childhood education/care",

    "BELONG":
        "Sense of belonging at school",

    "PERCOMP":
        "Perception of competitiveness",

    "PERCOOP":
        "Perception of cooperation",

    "ATTLNACT":
        "Attitude toward school / learning activities",

    "DISCLIMA":
        "Disciplinary climate",

    "TEACHSUP":
        "Teacher support",

    "DIRINS":
        "Teacher-directed instruction",

    "PERFEED":
        "Perceived feedback",

    "STIMREAD":
        "Teacher stimulation of reading engagement",

    "ADAPTIVITY":
        "Adaptive instruction",

    "TEACHINT":
        "Teacher interest / enthusiasm",

    "ESCS":
        "Economic, social and cultural status",

    "EDUSHORT":
        "Educational staff shortage",

    "RATCMP1":
        "Computer-related school resource ratio 1",

    "RATCMP2":
        "Computer-related school resource ratio 2",
}


registry_rows = []


for code in (
    EXPANDED_NONACHIEVEMENT
):

    registry_rows.append(
        {
            "Concept":
                code,

            "Construct":
                CONSTRUCT_LABELS.get(
                    code,
                    "Official wording to verify"
                ),

            "Level":
                LEVEL_MAP.get(
                    code,
                    "Individual"
                ),

            "Representation":
                "SingleColumn",

            "RawColumns":
                code,

            "FrozenCore14":
                code in CORE14,

            "ParentModule":
                code == "PA006Q09TA",
        }
    )


for concept, family in zip(
    ACHIEVEMENT_CONCEPTS,
    [
        "READ",
        "MATH",
        "SCIE"
    ]
):

    registry_rows.append(
        {
            "Concept":
                concept,

            "Construct":
                {
                    "READ":
                        "Reading achievement",

                    "MATH":
                        "Mathematics achievement",

                    "SCIE":
                        "Science achievement",
                }[
                    family
                ],

            "Level":
                "Individual",

            "Representation":
                "10 matched plausible values",

            "RawColumns":
                "|".join(
                    ACHIEVEMENT_FAMILIES[
                        family
                    ]
                ),

            "FrozenCore14":
                False,

            "ParentModule":
                False,
        }
    )


predictor_registry = pd.DataFrame(
    registry_rows
)


assert len(
    predictor_registry
) == 47


predictor_registry.to_csv(
    OUT_DIR
    / "expanded_predictor_registry.csv",
    index=False
)


# ============================================================
# 7. Determine the system universe from frozen OOF results
#
# Union across outcomes = systems that participated in at
# least one frozen primary model.
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


SYSTEMS = sorted(
    rankings[
        "Country"
    ]
    .dropna()
    .astype(
        str
    )
    .unique()
)


print(
    "\nFrozen model-system universe:",
    len(
        SYSTEMS
    )
)

print(
    SYSTEMS
)


# ============================================================
# 8. Search for likely processed per-system datasets
#
# We do NOT assume a path yet.
#
# Search likely project locations for tabular files.
# ============================================================

SEARCH_ROOTS = [
    Path("../data"),
    Path("../processed"),
    Path("../results"),
    Path(".."),
]


VALID_SUFFIXES = {
    ".pkl",
    ".pickle",
    ".parquet",
    ".feather",
}


all_candidate_files = []


for root in SEARCH_ROOTS:

    if not root.exists():

        continue


    for suffix in VALID_SUFFIXES:

        for path in root.rglob(
            f"*{suffix}"
        ):

            # skip our own new outputs
            if (
                "expanded_predictor_analysis"
                in str(
                    path
                )
            ):

                continue


            all_candidate_files.append(
                path.resolve()
            )


# Deduplicate paths

all_candidate_files = sorted(
    set(
        all_candidate_files
    )
)


print(
    "\nTabular binary files discovered:",
    len(
        all_candidate_files
    )
)


# ============================================================
# 9. Match filenames to PISA system codes
#
# Example acceptable names:
#
#   USA.pkl
#   USA_merged.pkl
#   processed_USA.parquet
#
# ============================================================

def filename_mentions_system(
    path,
    system
):

    name = (
        path.stem
        .upper()
    )


    pattern = (
        rf"(^|[^A-Z0-9])"
        rf"{re.escape(system.upper())}"
        rf"([^A-Z0-9]|$)"
    )


    return (
        re.search(
            pattern,
            name
        )
        is not None
    )


file_candidates_by_system = {
    system: []
    for system
    in SYSTEMS
}


for system in SYSTEMS:

    for path in all_candidate_files:

        if filename_mentions_system(
            path,
            system
        ):

            file_candidates_by_system[
                system
            ].append(
                path
            )


# ============================================================
# 10. Safe loader
# ============================================================

def load_tabular(
    path
):

    suffix = (
        path.suffix
        .lower()
    )


    if suffix in {
        ".pkl",
        ".pickle",
    }:

        obj = pd.read_pickle(
            path
        )


    elif suffix == ".parquet":

        obj = pd.read_parquet(
            path
        )


    elif suffix == ".feather":

        obj = pd.read_feather(
            path
        )


    else:

        return None


    if not isinstance(
        obj,
        pd.DataFrame
    ):

        return None


    return obj


# ============================================================
# 11. Score candidate files
#
# We only inspect schema + basic row count.
#
# High score =
# contains many intended expanded predictors and/or PVs.
# ============================================================

ALL_ACHIEVEMENT_RAW = (
    ACHIEVEMENT_FAMILIES[
        "READ"
    ]
    +
    ACHIEVEMENT_FAMILIES[
        "MATH"
    ]
    +
    ACHIEVEMENT_FAMILIES[
        "SCIE"
    ]
)


schema_rows = []


for system in SYSTEMS:

    candidates = (
        file_candidates_by_system[
            system
        ]
    )


    for path in candidates:

        try:

            df = load_tabular(
                path
            )


            if df is None:

                continue


            columns = set(
                map(
                    str,
                    df.columns
                )
            )


            n_expanded = sum(
                column in columns
                for column
                in EXPANDED_NONACHIEVEMENT
            )


            n_core = sum(
                column in columns
                for column
                in CORE14
            )


            n_pv = sum(
                column in columns
                for column
                in ALL_ACHIEVEMENT_RAW
            )


            schema_rows.append(
                {
                    "Country":
                        system,

                    "Path":
                        str(
                            path
                        ),

                    "Rows":
                        len(
                            df
                        ),

                    "Columns":
                        len(
                            df.columns
                        ),

                    "Core14Columns":
                        n_core,

                    "Expanded44Columns":
                        n_expanded,

                    "AchievementPVColumns":
                        n_pv,

                    "PA006Q09TA_Present":
                        (
                            "PA006Q09TA"
                            in columns
                        ),

                    # weighting strongly favors
                    # intended predictor coverage
                    "SchemaScore":
                        (
                            10
                            *
                            n_expanded
                            +
                            2
                            *
                            n_pv
                            +
                            n_core
                        ),
                }
            )


            del df

            gc.collect()


        except Exception as error:

            schema_rows.append(
                {
                    "Country":
                        system,

                    "Path":
                        str(
                            path
                        ),

                    "Rows":
                        np.nan,

                    "Columns":
                        np.nan,

                    "Core14Columns":
                        np.nan,

                    "Expanded44Columns":
                        np.nan,

                    "AchievementPVColumns":
                        np.nan,

                    "PA006Q09TA_Present":
                        np.nan,

                    "SchemaScore":
                        -1,

                    "LoadError":
                        str(
                            error
                        )[
                            :200
                        ],
                }
            )


schema_candidates = pd.DataFrame(
    schema_rows
)


schema_candidates.to_csv(
    OUT_DIR
    / "processed_file_schema_candidates.csv",
    index=False
)


# ============================================================
# 12. Pick best candidate file per system
# ============================================================

if len(
    schema_candidates
) > 0:

    usable_schema = (
        schema_candidates[
            schema_candidates[
                "SchemaScore"
            ]
            >=
            0
        ]
        .copy()
    )


    if len(
        usable_schema
    ) > 0:

        best_files = (
            usable_schema
            .sort_values(
                [
                    "Country",
                    "SchemaScore",
                    "Rows",
                ],
                ascending=[
                    True,
                    False,
                    False,
                ]
            )
            .groupby(
                "Country",
                as_index=False
            )
            .first()
        )


    else:

        best_files = pd.DataFrame()


else:

    best_files = pd.DataFrame()


if len(
    best_files
) > 0:

    best_files.to_csv(
        OUT_DIR
        / "best_processed_file_by_system.csv",
        index=False
    )


# ============================================================
# 13. Print registry
# ============================================================

print("\n")
print("=" * 120)
print("EXPANDED PREDICTOR REGISTRY")
print("=" * 120)


print(
    predictor_registry[
        [
            "Concept",
            "Level",
            "Representation",
            "FrozenCore14",
            "ParentModule",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 14. Print discovery result
# ============================================================

print("\n")
print("=" * 120)
print("PROCESSED DATA DISCOVERY")
print("=" * 120)


systems_with_filename_candidates = sum(
    len(
        file_candidates_by_system[
            system
        ]
    )
    >
    0
    for system
    in SYSTEMS
)


print(
    "Systems with filename-matched candidate files:",
    systems_with_filename_candidates,
    "/",
    len(
        SYSTEMS
    )
)


if len(
    best_files
) == 0:

    print(
        "\nNo usable per-system DataFrame files were "
        "identified automatically."
    )

    print(
        "\nDo NOT change anything yet."
    )

    print(
        "Send me this discovery output and we will point "
        "the next cell at the correct frozen dataset."
    )


else:

    print(
        "\nBest discovered file per system:"
    )


    print(
        best_files[
            [
                "Country",
                "Rows",
                "Columns",
                "Core14Columns",
                "Expanded44Columns",
                "AchievementPVColumns",
                "PA006Q09TA_Present",
                "SchemaScore",
                "Path",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ============================================================
# 15. Coverage summary of discovered schemas
# ============================================================

if len(
    best_files
) > 0:

    print("\n")
    print("=" * 120)
    print("DISCOVERED SCHEMA SUMMARY")
    print("=" * 120)


    print(
        "Systems discovered:",
        len(
            best_files
        )
    )


    print(
        "\nCore14 column coverage:"
    )


    print(
        best_files[
            "Core14Columns"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


    print(
        "\nExpanded44 column coverage:"
    )


    print(
        best_files[
            "Expanded44Columns"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


    print(
        "\nAchievement PV raw-column coverage:"
    )


    print(
        best_files[
            "AchievementPVColumns"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


    print(
        "\nPA006Q09TA structural availability:"
    )


    print(
        best_files[
            "PA006Q09TA_Present"
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


print("\n")
print("=" * 120)
print("PHASE 3A REGISTRY / DISCOVERY COMPLETE")
print("=" * 120)


print(
    "\nNo expanded models have been fitted."
)

Expanded non-achievement predictors: 44
Achievement concepts: 3
TOTAL conceptual predictors: 47

Frozen model-system universe: 75
['ALB', 'ARE', 'ARG', 'AUS', 'AUT', 'BEL', 'BGR', 'BIH', 'BLR', 'BRA', 'BRN', 'CAN', 'CHE', 'CHL', 'COL', 'CRI', 'CZE', 'DEU', 'DNK', 'DOM', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GEO', 'GRC', 'HKG', 'HRV', 'HUN', 'IDN', 'IRL', 'ISL', 'ITA', 'JOR', 'JPN', 'KAZ', 'KOR', 'KSV', 'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MEX', 'MLT', 'MNE', 'MYS', 'NLD', 'NOR', 'NZL', 'PAN', 'PER', 'PHL', 'POL', 'PRT', 'QAT', 'QAZ', 'QCI', 'ROU', 'RUS', 'SAU', 'SGP', 'SRB', 'SVK', 'SVN', 'SWE', 'TAP', 'THA', 'TUR', 'UKR', 'URY', 'USA', 'VNM']

Tabular binary files discovered: 647


EXPANDED PREDICTOR REGISTRY
   Concept           Level              Representation  FrozenCore14  ParentModule
 ST004D01T      Individual                SingleColumn          True         False
       AGE      Individual                SingleColumn          True         False
     GRADE      Individual 

In [2]:
# ============================================================
# PHASE 3B — VALUE-LEVEL COVERAGE + ELIGIBILITY FRONTIER
# ============================================================
#
# PURPOSE
# -------
# Structural presence was confirmed for all 47 concepts.
#
# Now determine whether those columns actually contain enough
# usable information within each education system.
#
#
# SINGLE-COLUMN CONCEPT IS USABLE IF:
#
#   1. numeric finite values exist
#   2. > 1 unique finite value
#   3. missingness <= threshold
#
#
# ACHIEVEMENT CONCEPT IS USABLE IF:
#
#   ALL 10 matched plausible values:
#
#       PViREAD
#       PViMATH
#       PViSCIE
#
#   individually satisfy:
#
#       missingness <= threshold
#       > 1 unique finite value
#
#
# WHY ALL 10?
# -----------
# The final expanded model will preserve the same
# plausible-value design as the frozen achievement analysis.
#
# We do NOT want a system where PV1 works but PV7 does not.
#
#
# PRIMARY FRONTIER:
#
#   47 = all intended predictors
#
#   46 = all except PA006Q09TA
#
# We also report:
#
#   44 = all non-achievement predictors
#   43 = non-achievement predictors excluding PA006Q09TA
#   14 = frozen core
#   3  = achievement families
#
#
# NO MODEL FITTING.
# ============================================================


from pathlib import Path
import gc

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


OUT_DIR = (
    Path("../results")
    / "expanded_predictor_analysis"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MISSINGNESS_THRESHOLDS = [
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
]


PRIMARY_THRESHOLD = 0.50


# ============================================================
# 2. Confirm variables from Phase 3A exist
# ============================================================

required_objects = [
    "SYSTEMS",
    "CORE14",
    "EXPANDED_NONACHIEVEMENT",
    "ACHIEVEMENT_FAMILIES",
    "ACHIEVEMENT_CONCEPTS",
]


for object_name in required_objects:

    if object_name not in globals():

        raise RuntimeError(
            f"{object_name} not found. "
            "Run the Phase 3A registry/discovery cell first."
        )


assert len(
    EXPANDED_NONACHIEVEMENT
) == 44


assert len(
    ACHIEVEMENT_CONCEPTS
) == 3


assert (
    len(
        EXPANDED_NONACHIEVEMENT
    )
    +
    len(
        ACHIEVEMENT_CONCEPTS
    )
    ==
    47
)


# ============================================================
# 3. Concept universes
# ============================================================

FULL47 = (
    EXPANDED_NONACHIEVEMENT
    +
    ACHIEVEMENT_CONCEPTS
)


NO_PARENT46 = [
    concept
    for concept
    in FULL47
    if concept
    !=
    "PA006Q09TA"
]


NONACH44 = (
    EXPANDED_NONACHIEVEMENT.copy()
)


NONACH43_NO_PARENT = [
    concept
    for concept
    in EXPANDED_NONACHIEVEMENT
    if concept
    !=
    "PA006Q09TA"
]


ACHIEVEMENT3 = (
    ACHIEVEMENT_CONCEPTS.copy()
)


assert len(
    FULL47
) == 47


assert len(
    NO_PARENT46
) == 46


assert len(
    NONACH44
) == 44


assert len(
    NONACH43_NO_PARENT
) == 43


assert len(
    CORE14
) == 14


assert len(
    ACHIEVEMENT3
) == 3


# ============================================================
# 4. Helpers
# ============================================================

def numeric_audit(
    series
):

    original_nonmissing = int(
        series.notna().sum()
    )


    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )


    values = numeric.to_numpy(
        dtype=float
    )


    finite_mask = np.isfinite(
        values
    )


    finite_values = values[
        finite_mask
    ]


    n_total = len(
        series
    )


    n_valid = int(
        finite_mask.sum()
    )


    missing_rate = (
        1.0
        -
        (
            n_valid
            /
            n_total
        )
        if n_total
        >
        0
        else np.nan
    )


    unique_count = (
        int(
            pd.Series(
                finite_values
            )
            .nunique()
        )
        if n_valid
        >
        0
        else 0
    )


    coercion_loss = (
        original_nonmissing
        -
        n_valid
    )


    minimum = (
        float(
            np.min(
                finite_values
            )
        )
        if n_valid
        >
        0
        else np.nan
    )


    maximum = (
        float(
            np.max(
                finite_values
            )
        )
        if n_valid
        >
        0
        else np.nan
    )


    return {

        "N_Total":
            n_total,

        "N_Valid":
            n_valid,

        "MissingRate":
            missing_rate,

        "UniqueFinite":
            unique_count,

        "OriginalNonMissing":
            original_nonmissing,

        "NonNumericOrNonFiniteLoss":
            coercion_loss,

        "ObservedMin":
            minimum,

        "ObservedMax":
            maximum,
    }


# ============================================================
# 5. Audit all single-column concepts
# ============================================================

audit_rows = []

pv_detail_rows = []


for system_index, country in enumerate(
    SYSTEMS,
    start=1
):

    path = (
        PROCESSED_DIR
        /
        f"{country}_analysis.pkl"
    )


    if not path.exists():

        raise FileNotFoundError(
            f"Missing processed file: {path}"
        )


    df = pd.read_pickle(
        path
    )


    print(
        f"[{system_index:02d}/{len(SYSTEMS)}] "
        f"{country}: {len(df):,} rows"
    )


    # --------------------------------------------------------
    # 5A. Ordinary single-column concepts
    # --------------------------------------------------------

    for concept in (
        EXPANDED_NONACHIEVEMENT
    ):

        if concept not in df.columns:

            audit_rows.append(
                {
                    "Country":
                        country,

                    "Concept":
                        concept,

                    "ConceptType":
                        "SingleColumn",

                    "StructuralPresent":
                        False,

                    "N_Total":
                        len(
                            df
                        ),

                    "N_Valid":
                        0,

                    "MissingRate":
                        1.0,

                    "UniqueFinite":
                        0,

                    "OriginalNonMissing":
                        0,

                    "NonNumericOrNonFiniteLoss":
                        np.nan,

                    "ObservedMin":
                        np.nan,

                    "ObservedMax":
                        np.nan,
                }
            )


            continue


        result = numeric_audit(
            df[
                concept
            ]
        )


        audit_rows.append(
            {
                "Country":
                    country,

                "Concept":
                    concept,

                "ConceptType":
                    "SingleColumn",

                "StructuralPresent":
                    True,

                **result,
            }
        )


    # --------------------------------------------------------
    # 5B. Achievement families
    # --------------------------------------------------------

    for family_name, pv_columns in (
        ACHIEVEMENT_FAMILIES.items()
    ):

        concept = (
            f"PV_{family_name}"
        )


        pv_stats = []


        for pv_index, pv_column in enumerate(
            pv_columns,
            start=1
        ):

            if pv_column not in df.columns:

                pv_result = {

                    "N_Total":
                        len(
                            df
                        ),

                    "N_Valid":
                        0,

                    "MissingRate":
                        1.0,

                    "UniqueFinite":
                        0,

                    "OriginalNonMissing":
                        0,

                    "NonNumericOrNonFiniteLoss":
                        np.nan,

                    "ObservedMin":
                        np.nan,

                    "ObservedMax":
                        np.nan,
                }


                structural_present = False


            else:

                pv_result = numeric_audit(
                    df[
                        pv_column
                    ]
                )


                structural_present = True


            pv_stats.append(
                pv_result
            )


            pv_detail_rows.append(
                {
                    "Country":
                        country,

                    "Concept":
                        concept,

                    "PV":
                        pv_index,

                    "RawColumn":
                        pv_column,

                    "StructuralPresent":
                        structural_present,

                    **pv_result,
                }
            )


        pv_missing_rates = np.array(
            [
                stat[
                    "MissingRate"
                ]
                for stat
                in pv_stats
            ],
            dtype=float
        )


        pv_unique_counts = np.array(
            [
                stat[
                    "UniqueFinite"
                ]
                for stat
                in pv_stats
            ],
            dtype=float
        )


        pv_valid_counts = np.array(
            [
                stat[
                    "N_Valid"
                ]
                for stat
                in pv_stats
            ],
            dtype=float
        )


        all_present = all(
            column
            in df.columns
            for column
            in pv_columns
        )


        # Conservative concept-level missingness:
        #
        # worst of the 10 matched PVs.
        concept_missing = float(
            np.max(
                pv_missing_rates
            )
        )


        concept_unique = int(
            np.min(
                pv_unique_counts
            )
        )


        concept_valid = int(
            np.min(
                pv_valid_counts
            )
        )


        audit_rows.append(
            {
                "Country":
                    country,

                "Concept":
                    concept,

                "ConceptType":
                    "AchievementFamily",

                "StructuralPresent":
                    all_present,

                "N_Total":
                    len(
                        df
                    ),

                "N_Valid":
                    concept_valid,

                "MissingRate":
                    concept_missing,

                "MeanPVMissingRate":
                    float(
                        np.mean(
                            pv_missing_rates
                        )
                    ),

                "BestPVMissingRate":
                    float(
                        np.min(
                            pv_missing_rates
                        )
                    ),

                "WorstPVMissingRate":
                    concept_missing,

                "UniqueFinite":
                    concept_unique,

                "MinPVUniqueFinite":
                    concept_unique,

                "OriginalNonMissing":
                    np.nan,

                "NonNumericOrNonFiniteLoss":
                    int(
                        np.sum(
                            [
                                stat[
                                    "NonNumericOrNonFiniteLoss"
                                ]
                                for stat
                                in pv_stats
                                if pd.notna(
                                    stat[
                                        "NonNumericOrNonFiniteLoss"
                                    ]
                                )
                            ]
                        )
                    ),

                "ObservedMin":
                    float(
                        np.nanmin(
                            [
                                stat[
                                    "ObservedMin"
                                ]
                                for stat
                                in pv_stats
                            ]
                        )
                    )
                    if concept_valid
                    >
                    0
                    else np.nan,

                "ObservedMax":
                    float(
                        np.nanmax(
                            [
                                stat[
                                    "ObservedMax"
                                ]
                                for stat
                                in pv_stats
                            ]
                        )
                    )
                    if concept_valid
                    >
                    0
                    else np.nan,
            }
        )


    del df

    gc.collect()


concept_audit = pd.DataFrame(
    audit_rows
)


pv_detail_audit = pd.DataFrame(
    pv_detail_rows
)


# ============================================================
# 6. Add usability flags at every threshold
# ============================================================

for threshold in (
    MISSINGNESS_THRESHOLDS
):

    label = int(
        threshold
        *
        100
    )


    concept_audit[
        f"Usable_MaxMissing{label}"
    ] = (

        concept_audit[
            "StructuralPresent"
        ].fillna(
            False
        )

        &

        (
            concept_audit[
                "MissingRate"
            ]
            <=
            threshold
        )

        &

        (
            concept_audit[
                "UniqueFinite"
            ]
            >
            1
        )
    )


    # PV detail level
    pv_detail_audit[
        f"Usable_MaxMissing{label}"
    ] = (

        pv_detail_audit[
            "StructuralPresent"
        ].fillna(
            False
        )

        &

        (
            pv_detail_audit[
                "MissingRate"
            ]
            <=
            threshold
        )

        &

        (
            pv_detail_audit[
                "UniqueFinite"
            ]
            >
            1
        )
    )


# ============================================================
# 7. Save raw audits
# ============================================================

concept_audit.to_csv(
    OUT_DIR
    / "expanded_concept_value_audit.csv",
    index=False
)


pv_detail_audit.to_csv(
    OUT_DIR
    / "expanded_achievement_pv_detail_audit.csv",
    index=False
)


# ============================================================
# 8. QA dimensions
# ============================================================

expected_rows = (
    len(
        SYSTEMS
    )
    *
    47
)


if len(
    concept_audit
) != expected_rows:

    raise RuntimeError(
        "Unexpected audit row count: "
        f"{len(concept_audit)} "
        f"vs expected {expected_rows}"
    )


if (
    concept_audit[
        [
            "Country",
            "Concept"
        ]
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate Country × Concept audit rows found."
    )


# ============================================================
# 9. Concept-level international coverage summary
# ============================================================

concept_summary_rows = []


for concept in FULL47:

    temp = (
        concept_audit[
            concept_audit[
                "Concept"
            ]
            ==
            concept
        ]
    )


    row = {

        "Concept":
            concept,

        "ConceptType":
            temp[
                "ConceptType"
            ]
            .iloc[
                0
            ],

        "N_Systems":
            len(
                temp
            ),

        "N_StructuralPresent":
            int(
                temp[
                    "StructuralPresent"
                ]
                .sum()
            ),

        "MedianMissingRate":
            temp[
                "MissingRate"
            ]
            .median(),

        "MeanMissingRate":
            temp[
                "MissingRate"
            ]
            .mean(),

        "MaxMissingRate":
            temp[
                "MissingRate"
            ]
            .max(),

        "MinUniqueFinite":
            temp[
                "UniqueFinite"
            ]
            .min(),
    }


    for threshold in (
        MISSINGNESS_THRESHOLDS
    ):

        label = int(
            threshold
            *
            100
        )


        row[
            f"N_Usable_{label}"
        ] = int(
            temp[
                f"Usable_MaxMissing{label}"
            ]
            .sum()
        )


    concept_summary_rows.append(
        row
    )


concept_summary = pd.DataFrame(
    concept_summary_rows
)


concept_summary = (
    concept_summary
    .sort_values(
        [
            "N_Usable_50",
            "MaxMissingRate",
            "Concept"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
)


concept_summary.to_csv(
    OUT_DIR
    / "expanded_concept_international_coverage.csv",
    index=False
)


# ============================================================
# 10. System-level concept counts
# ============================================================

system_summary_rows = []


for country in SYSTEMS:

    temp = (
        concept_audit[
            concept_audit[
                "Country"
            ]
            ==
            country
        ]
    )


    row = {
        "Country":
            country
    }


    for threshold in (
        MISSINGNESS_THRESHOLDS
    ):

        label = int(
            threshold
            *
            100
        )


        usable_col = (
            f"Usable_MaxMissing{label}"
        )


        row[
            f"N_Usable47_{label}"
        ] = int(
            temp[
                usable_col
            ]
            .sum()
        )


        row[
            f"N_Usable46NoParent_{label}"
        ] = int(
            temp[
                temp[
                    "Concept"
                ]
                .isin(
                    NO_PARENT46
                )
            ][
                usable_col
            ]
            .sum()
        )


        row[
            f"N_Usable44NonAch_{label}"
        ] = int(
            temp[
                temp[
                    "Concept"
                ]
                .isin(
                    NONACH44
                )
            ][
                usable_col
            ]
            .sum()
        )


        row[
            f"N_Usable43NoParentNonAch_{label}"
        ] = int(
            temp[
                temp[
                    "Concept"
                ]
                .isin(
                    NONACH43_NO_PARENT
                )
            ][
                usable_col
            ]
            .sum()
        )


        row[
            f"N_UsableCore14_{label}"
        ] = int(
            temp[
                temp[
                    "Concept"
                ]
                .isin(
                    CORE14
                )
            ][
                usable_col
            ]
            .sum()
        )


        row[
            f"N_UsableAchievement3_{label}"
        ] = int(
            temp[
                temp[
                    "Concept"
                ]
                .isin(
                    ACHIEVEMENT3
                )
            ][
                usable_col
            ]
            .sum()
        )


    system_summary_rows.append(
        row
    )


system_summary = pd.DataFrame(
    system_summary_rows
)


system_summary.to_csv(
    OUT_DIR
    / "expanded_system_coverage_counts.csv",
    index=False
)


# ============================================================
# 11. Eligibility helper
# ============================================================

def eligible_systems_for_universe(
    concepts,
    threshold
):

    label = int(
        threshold
        *
        100
    )


    usable_col = (
        f"Usable_MaxMissing{label}"
    )


    eligible = []


    for country in SYSTEMS:

        temp = (
            concept_audit[
                (
                    concept_audit[
                        "Country"
                    ]
                    ==
                    country
                )
                &
                (
                    concept_audit[
                        "Concept"
                    ]
                    .isin(
                        concepts
                    )
                )
            ]
        )


        if len(
            temp
        ) != len(
            concepts
        ):

            continue


        if temp[
            usable_col
        ].all():

            eligible.append(
                country
            )


    return sorted(
        eligible
    )


# ============================================================
# 12. Cohort eligibility frontier
# ============================================================

UNIVERSES = {

    "Full47":
        FULL47,

    "NoParent46":
        NO_PARENT46,

    "NonAchievement44":
        NONACH44,

    "NonAchievement43_NoParent":
        NONACH43_NO_PARENT,

    "Core14":
        CORE14,

    "Achievement3":
        ACHIEVEMENT3,
}


frontier_rows = []


cohort_lists = {}


for threshold in (
    MISSINGNESS_THRESHOLDS
):

    label = int(
        threshold
        *
        100
    )


    for universe_name, concepts in (
        UNIVERSES.items()
    ):

        eligible = (
            eligible_systems_for_universe(
                concepts,
                threshold
            )
        )


        cohort_lists[
            (
                universe_name,
                label
            )
        ] = eligible


        frontier_rows.append(
            {
                "Universe":
                    universe_name,

                "N_Concepts":
                    len(
                        concepts
                    ),

                "MaxMissingness":
                    threshold,

                "N_EligibleSystems":
                    len(
                        eligible
                    ),

                "EligibleSystems":
                    "|".join(
                        eligible
                    ),
            }
        )


eligibility_frontier = pd.DataFrame(
    frontier_rows
)


eligibility_frontier.to_csv(
    OUT_DIR
    / "expanded_eligibility_frontier.csv",
    index=False
)


# ============================================================
# 13. Common predictor frontier across ALL 75 systems
#
# This is a separate question:
#
# At each threshold, which expanded concepts satisfy the
# usability criterion in EVERY system?
#
# Useful for understanding why the original strict
# international predictor set became much smaller.
# ============================================================

common_predictor_rows = []


for threshold in (
    MISSINGNESS_THRESHOLDS
):

    label = int(
        threshold
        *
        100
    )


    usable_col = (
        f"Usable_MaxMissing{label}"
    )


    common_concepts = []


    for concept in FULL47:

        temp = (
            concept_audit[
                concept_audit[
                    "Concept"
                ]
                ==
                concept
            ]
        )


        if (
            len(
                temp
            )
            ==
            len(
                SYSTEMS
            )
            and
            temp[
                usable_col
            ]
            .all()
        ):

            common_concepts.append(
                concept
            )


    common_predictor_rows.append(
        {
            "MaxMissingness":
                threshold,

            "N_CommonConcepts_All75":
                len(
                    common_concepts
                ),

            "CommonConcepts":
                "|".join(
                    common_concepts
                ),
        }
    )


common_predictor_frontier = pd.DataFrame(
    common_predictor_rows
)


common_predictor_frontier.to_csv(
    OUT_DIR
    / "expanded_common_predictor_frontier_all75.csv",
    index=False
)


# ============================================================
# 14. Primary 50% eligibility
# ============================================================

FULL47_50 = (
    cohort_lists[
        (
            "Full47",
            50
        )
    ]
)


NO_PARENT46_50 = (
    cohort_lists[
        (
            "NoParent46",
            50
        )
    ]
)


NONACH44_50 = (
    cohort_lists[
        (
            "NonAchievement44",
            50
        )
    ]
)


NONACH43_50 = (
    cohort_lists[
        (
            "NonAchievement43_NoParent",
            50
        )
    ]
)


CORE14_50 = (
    cohort_lists[
        (
            "Core14",
            50
        )
    ]
)


ACH3_50 = (
    cohort_lists[
        (
            "Achievement3",
            50
        )
    ]
)


# ============================================================
# 15. Identify exactly which concepts block each system
# at the primary 50% threshold
# ============================================================

blocking_rows = []


for country in SYSTEMS:

    temp = (
        concept_audit[
            concept_audit[
                "Country"
            ]
            ==
            country
        ]
    )


    blocked = (
        temp[
            ~temp[
                "Usable_MaxMissing50"
            ]
        ][
            "Concept"
        ]
        .tolist()
    )


    blocking_rows.append(
        {
            "Country":
                country,

            "N_BlockedConcepts_50":
                len(
                    blocked
                ),

            "BlockedConcepts_50":
                "|".join(
                    blocked
                ),
        }
    )


blocking_by_system = pd.DataFrame(
    blocking_rows
)


blocking_by_system = (
    blocking_by_system
    .sort_values(
        [
            "N_BlockedConcepts_50",
            "Country"
        ]
    )
)


blocking_by_system.to_csv(
    OUT_DIR
    / "expanded_blocking_concepts_by_system_50pct.csv",
    index=False
)


# ============================================================
# 16. Which concepts are the major bottlenecks at 50%?
# ============================================================

bottleneck_50 = (
    concept_summary[
        [
            "Concept",
            "ConceptType",
            "N_Usable_50",
            "MedianMissingRate",
            "MeanMissingRate",
            "MaxMissingRate",
            "MinUniqueFinite",
        ]
    ]
    .sort_values(
        [
            "N_Usable_50",
            "MeanMissingRate"
        ],
        ascending=[
            True,
            False
        ]
    )
)


bottleneck_50.to_csv(
    OUT_DIR
    / "expanded_bottlenecks_50pct.csv",
    index=False
)


# ============================================================
# 17. Numeric / formatting QA
# ============================================================

format_problems = (
    concept_audit[
        concept_audit[
            "NonNumericOrNonFiniteLoss"
        ]
        .fillna(
            0
        )
        >
        0
    ][
        [
            "Country",
            "Concept",
            "ConceptType",
            "OriginalNonMissing",
            "N_Valid",
            "NonNumericOrNonFiniteLoss",
        ]
    ]
)


format_problems.to_csv(
    OUT_DIR
    / "expanded_numeric_format_problems.csv",
    index=False
)


# ============================================================
# 18. Raw item-code QA
#
# We don't automatically alter anything.
#
# This merely prints observed ranges so we can catch an
# obvious retained missing-code problem before modeling.
# ============================================================

RAW_ITEM_CODES = [

    "ST184Q01HA",
    "ST185Q01HA",

    "ST206Q01HA",

    "ST207Q01HA",
    "ST207Q02HA",
    "ST207Q03HA",
    "ST207Q04HA",
    "ST207Q05HA",

    "PA006Q09TA",
]


raw_item_range = (
    concept_audit[
        concept_audit[
            "Concept"
        ]
        .isin(
            RAW_ITEM_CODES
        )
    ]
    .groupby(
        "Concept",
        as_index=False
    )
    .agg(
        NSystems=(
            "Country",
            "nunique"
        ),

        MinObserved=(
            "ObservedMin",
            "min"
        ),

        MaxObserved=(
            "ObservedMax",
            "max"
        ),

        MedianMissing=(
            "MissingRate",
            "median"
        ),

        MaxMissing=(
            "MissingRate",
            "max"
        ),

        NUsable50=(
            "Usable_MaxMissing50",
            "sum"
        ),
    )
)


raw_item_range.to_csv(
    OUT_DIR
    / "expanded_raw_item_range_qa.csv",
    index=False
)


# ============================================================
# 19. Print international concept coverage
# ============================================================

print("\n")
print("=" * 125)
print("CONCEPT COVERAGE AT PRIMARY 50% MISSINGNESS THRESHOLD")
print("=" * 125)


print(
    concept_summary[
        [
            "Concept",
            "ConceptType",
            "N_Usable_20",
            "N_Usable_30",
            "N_Usable_40",
            "N_Usable_50",
            "N_Usable_60",
            "N_Usable_70",
            "MedianMissingRate",
            "MaxMissingRate",
            "MinUniqueFinite",
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 20. Print cohort frontier
# ============================================================

print("\n")
print("=" * 125)
print("EXPANDED COHORT ELIGIBILITY FRONTIER")
print("=" * 125)


frontier_pivot = (
    eligibility_frontier
    .pivot(
        index="Universe",
        columns="MaxMissingness",
        values="N_EligibleSystems"
    )
)


frontier_pivot.columns = [
    f"{int(column * 100)}%"
    for column
    in frontier_pivot.columns
]


print(
    frontier_pivot
    .to_string()
)


# ============================================================
# 21. Primary 50% cohort lists
# ============================================================

print("\n")
print("=" * 125)
print("PRIMARY 50% EXPANDED COHORTS")
print("=" * 125)


print(
    "\nFULL 47:"
)

print(
    "N =",
    len(
        FULL47_50
    )
)

print(
    FULL47_50
)


print(
    "\nNO-PARENT 46:"
)

print(
    "N =",
    len(
        NO_PARENT46_50
    )
)

print(
    NO_PARENT46_50
)


print(
    "\nNON-ACHIEVEMENT 44:"
)

print(
    "N =",
    len(
        NONACH44_50
    )
)

print(
    NONACH44_50
)


print(
    "\nNON-ACHIEVEMENT 43, NO PARENT:"
)

print(
    "N =",
    len(
        NONACH43_50
    )
)

print(
    NONACH43_50
)


print(
    "\nACHIEVEMENT 3:"
)

print(
    "N =",
    len(
        ACH3_50
    )
)

print(
    ACH3_50
)


print(
    "\nCORE 14:"
)

print(
    "N =",
    len(
        CORE14_50
    )
)


# ============================================================
# 22. Major bottlenecks
# ============================================================

print("\n")
print("=" * 125)
print("MAJOR 50% COVERAGE BOTTLENECKS")
print("=" * 125)


print(
    bottleneck_50
    .head(
        20
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 23. Systems closest to full 47
# ============================================================

print("\n")
print("=" * 125)
print("SYSTEMS CLOSEST TO FULL 47 AT 50%")
print("=" * 125)


print(
    blocking_by_system
    .head(
        30
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 24. Achievement-family audit
# ============================================================

print("\n")
print("=" * 125)
print("ACHIEVEMENT FAMILY USABILITY AT 50%")
print("=" * 125)


achievement_summary = (
    concept_audit[
        concept_audit[
            "Concept"
        ]
        .isin(
            ACHIEVEMENT3
        )
    ]
    .pivot(
        index="Country",
        columns="Concept",
        values="Usable_MaxMissing50"
    )
)


print(
    achievement_summary
    .astype(
        int
    )
    .to_string()
)


# ============================================================
# 25. PA006 audit specifically
# ============================================================

print("\n")
print("=" * 125)
print("PA006Q09TA VALUE-LEVEL AVAILABILITY")
print("=" * 125)


pa006 = (
    concept_audit[
        concept_audit[
            "Concept"
        ]
        ==
        "PA006Q09TA"
    ][
        [
            "Country",
            "N_Valid",
            "MissingRate",
            "UniqueFinite",
            "Usable_MaxMissing50",
        ]
    ]
    .sort_values(
        [
            "Usable_MaxMissing50",
            "MissingRate"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print(
    pa006
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 26. Common predictor frontier across all 75 systems
# ============================================================

print("\n")
print("=" * 125)
print("COMMON PREDICTOR FRONTIER ACROSS ALL 75 SYSTEMS")
print("=" * 125)


print(
    common_predictor_frontier
    .to_string(
        index=False
    )
)


# ============================================================
# 27. Formatting QA
# ============================================================

print("\n")
print("=" * 125)
print("NUMERIC FORMAT QA")
print("=" * 125)


print(
    "Country × concept rows with nonnumeric/nonfinite "
    "loss:",
    len(
        format_problems
    )
)


if len(
    format_problems
) > 0:

    print(
        format_problems
        .head(
            30
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 28. Raw item-range QA
# ============================================================

print("\n")
print("=" * 125)
print("RAW ITEM RANGE QA")
print("=" * 125)


print(
    raw_item_range
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 29. Final stage status
# ============================================================

print("\n")
print("=" * 125)
print("PHASE 3B VALUE-LEVEL COVERAGE AUDIT COMPLETE")
print("=" * 125)


print(
    "\nNo expanded models have been fitted."
)


print(
    "\nPrimary questions for the next decision:"
)


print(
    "1. How many systems support the literal Full47?"
)


print(
    "2. How many support the NoParent46 universe?"
)


print(
    "3. Which predictors cause the largest cohort loss?"
)


print(
    "4. Do the achievement families behave as expected?"
)

[01/75] ALB: 6,359 rows
[02/75] ARE: 19,277 rows
[03/75] ARG: 11,975 rows
[04/75] AUS: 14,273 rows
[05/75] AUT: 6,802 rows
[06/75] BEL: 8,475 rows
[07/75] BGR: 5,294 rows
[08/75] BIH: 6,480 rows
[09/75] BLR: 5,803 rows
[10/75] BRA: 10,691 rows
[11/75] BRN: 6,828 rows
[12/75] CAN: 22,653 rows
[13/75] CHE: 5,822 rows
[14/75] CHL: 7,621 rows
[15/75] COL: 7,522 rows
[16/75] CRI: 7,221 rows
[17/75] CZE: 7,019 rows
[18/75] DEU: 5,451 rows
[19/75] DNK: 7,657 rows
[20/75] DOM: 5,674 rows
[21/75] ESP: 35,943 rows
[22/75] EST: 5,316 rows
[23/75] FIN: 5,649 rows
[24/75] FRA: 6,308 rows
[25/75] GBR: 13,818 rows
[26/75] GEO: 5,572 rows
[27/75] GRC: 6,403 rows
[28/75] HKG: 6,037 rows
[29/75] HRV: 6,609 rows
[30/75] HUN: 5,132 rows
[31/75] IDN: 12,098 rows
[32/75] IRL: 5,577 rows
[33/75] ISL: 3,296 rows
[34/75] ITA: 11,785 rows
[35/75] JOR: 8,963 rows
[36/75] JPN: 6,109 rows
[37/75] KAZ: 19,507 rows
[38/75] KOR: 6,650 rows
[39/75] KSV: 5,058 rows
[40/75] LTU: 6,885 rows
[41/75] LUX: 5,230 rows
[42/75

In [3]:
# ============================================================
# PHASE 3C — FREEZE EXPANDED COHORTS + OUTCOME SAMPLE AUDIT
# ============================================================
#
# PURPOSE
# -------
# Freeze the exact systems and student samples that will be
# used for the matched expanded-model comparisons.
#
#
# PRIMARY COMPARISONS
# -------------------
#
# A. NON-ACHIEVEMENT
#
#       Core14
#          vs
#       Expanded43
#
#    Same 53 systems
#    Same outcome-valid students
#    Same school folds
#
#
# B. ACHIEVEMENT-CONTROLLED
#
#       Core17
#          vs
#       Expanded46
#
#    where Core17 =
#
#       Core14
#       + PViREAD
#       + PViMATH
#       + PViSCIE
#
#    Same 53 systems
#    Same outcome-valid students
#    Same school folds
#    Same plausible-value replicate i
#
#
# C. PARENT-MODULE SENSITIVITY
#
#       Expanded46
#          vs
#       Full47
#
#    Same 12 systems
#    Same outcome-valid students
#    Same folds
#
#
# NO XGBOOST FITTING YET.
# ============================================================


from pathlib import Path
import gc

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


OUT_DIR = (
    Path("../results")
    / "expanded_predictor_analysis"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


# ============================================================
# 2. Freeze predictor universes
# ============================================================

EXPANDED43 = [
    predictor
    for predictor
    in EXPANDED_NONACHIEVEMENT
    if predictor
    !=
    "PA006Q09TA"
]


assert len(
    EXPANDED43
) == 43


CORE17_CONCEPTS = (
    CORE14
    +
    [
        "PV_READ",
        "PV_MATH",
        "PV_SCIE",
    ]
)


EXPANDED46_CONCEPTS = (
    EXPANDED43
    +
    [
        "PV_READ",
        "PV_MATH",
        "PV_SCIE",
    ]
)


FULL47_CONCEPTS = (
    EXPANDED46_CONCEPTS
    +
    [
        "PA006Q09TA"
    ]
)


assert len(
    CORE14
) == 14

assert len(
    CORE17_CONCEPTS
) == 17

assert len(
    EXPANDED43
) == 43

assert len(
    EXPANDED46_CONCEPTS
) == 46

assert len(
    FULL47_CONCEPTS
) == 47


# ============================================================
# 3. Freeze system cohorts from Phase 3B
# ============================================================

PRIMARY53 = sorted(
    NO_PARENT46_50
)


FULL_PARENT12 = sorted(
    FULL47_50
)


assert len(
    PRIMARY53
) == 53


assert len(
    FULL_PARENT12
) == 12


print(
    "Primary expanded cohort:",
    len(
        PRIMARY53
    ),
    "systems"
)


print(
    PRIMARY53
)


print(
    "\nLiteral Full47 parent-module cohort:",
    len(
        FULL_PARENT12
    ),
    "systems"
)


print(
    FULL_PARENT12
)


# ============================================================
# 4. Save frozen cohort definitions
# ============================================================

cohort_definition_rows = []


for country in PRIMARY53:

    cohort_definition_rows.append(
        {
            "Country":
                country,

            "Primary53":
                True,

            "FullParent12":
                (
                    country
                    in
                    FULL_PARENT12
                ),
        }
    )


for country in FULL_PARENT12:

    if country not in PRIMARY53:

        cohort_definition_rows.append(
            {
                "Country":
                    country,

                "Primary53":
                    False,

                "FullParent12":
                    True,
            }
        )


frozen_cohorts = pd.DataFrame(
    cohort_definition_rows
)


frozen_cohorts.to_csv(
    OUT_DIR
    / "frozen_expanded_system_cohorts.csv",
    index=False
)


# ============================================================
# 5. Utility: identify key columns safely
# ============================================================

SCHOOL_ID_CANDIDATES = [
    "CNTSCHID",
    "SCHOOLID",
    "SchoolID",
]


STUDENT_ID_CANDIDATES = [
    "CNTSTUID",
    "STUDENTID",
    "StudentID",
]


WEIGHT_CANDIDATES = [
    "W_FSTUWT",
    "W_FSTURWT",
    "STUWT",
]


def first_existing(
    columns,
    candidates
):

    for candidate in candidates:

        if candidate in columns:

            return candidate


    return None


# ============================================================
# 6. Inspect one file and resolve identifiers
# ============================================================

reference_country = (
    PRIMARY53[
        0
    ]
)


reference_path = (
    PROCESSED_DIR
    /
    f"{reference_country}_analysis.pkl"
)


reference_df = pd.read_pickle(
    reference_path
)


school_col = first_existing(
    reference_df.columns,
    SCHOOL_ID_CANDIDATES
)


student_col = first_existing(
    reference_df.columns,
    STUDENT_ID_CANDIDATES
)


weight_col = first_existing(
    reference_df.columns,
    WEIGHT_CANDIDATES
)


print(
    "\nResolved school ID:",
    school_col
)


print(
    "Resolved student ID:",
    student_col
)


print(
    "Resolved student weight:",
    weight_col
)


if school_col is None:

    raise RuntimeError(
        "Could not locate school ID column."
    )


if weight_col is None:

    raise RuntimeError(
        "Could not locate student weight column."
    )


missing_outcome_columns = [
    outcome
    for outcome
    in OUTCOMES
    if outcome
    not in reference_df.columns
]


if len(
    missing_outcome_columns
) > 0:

    print(
        "\nOutcome columns not directly found:",
        missing_outcome_columns
    )


    print(
        "\nColumns containing BULL / ST038:"
    )


    print(
        [
            column
            for column
            in reference_df.columns
            if (
                "BULL"
                in str(
                    column
                ).upper()
                or
                "ST038"
                in str(
                    column
                ).upper()
            )
        ]
    )


    raise RuntimeError(
        "Outcome column names need resolution before modeling."
    )


del reference_df

gc.collect()


# ============================================================
# 7. Validate schema consistency across PRIMARY53
# ============================================================

schema_rows = []


required_nonachievement_columns = sorted(
    set(
        CORE14
        +
        EXPANDED43
    )
)


required_pv_columns = sorted(
    set(
        ACHIEVEMENT_FAMILIES[
            "READ"
        ]
        +
        ACHIEVEMENT_FAMILIES[
            "MATH"
        ]
        +
        ACHIEVEMENT_FAMILIES[
            "SCIE"
        ]
    )
)


for country in PRIMARY53:

    path = (
        PROCESSED_DIR
        /
        f"{country}_analysis.pkl"
    )


    df = pd.read_pickle(
        path
    )


    columns = set(
        df.columns
    )


    schema_rows.append(
        {
            "Country":
                country,

            "Rows":
                len(
                    df
                ),

            "SchoolColumnPresent":
                school_col
                in columns,

            "WeightColumnPresent":
                weight_col
                in columns,

            "StudentColumnPresent":
                (
                    student_col
                    in columns
                    if student_col
                    is not None
                    else np.nan
                ),

            "AllOutcomesPresent":
                all(
                    outcome
                    in columns
                    for outcome
                    in OUTCOMES
                ),

            "AllCore14Present":
                all(
                    predictor
                    in columns
                    for predictor
                    in CORE14
                ),

            "AllExpanded43Present":
                all(
                    predictor
                    in columns
                    for predictor
                    in EXPANDED43
                ),

            "All30PVColumnsPresent":
                all(
                    column
                    in columns
                    for column
                    in required_pv_columns
                ),
        }
    )


    del df

    gc.collect()


schema_check = pd.DataFrame(
    schema_rows
)


schema_check.to_csv(
    OUT_DIR
    / "expanded_primary53_schema_check.csv",
    index=False
)


# ============================================================
# 8. Outcome-level sample audit
#
# Sample definition:
#
# valid outcome
# valid positive survey weight
# nonmissing school ID
#
# Predictors are NOT used to remove rows.
#
# XGBoost handles predictor missingness natively.
#
# This is critical because it ensures Core and Expanded models
# are evaluated on the exact same students.
# ============================================================

sample_rows = []


for system_index, country in enumerate(
    PRIMARY53,
    start=1
):

    path = (
        PROCESSED_DIR
        /
        f"{country}_analysis.pkl"
    )


    df = pd.read_pickle(
        path
    )


    print(
        f"[{system_index:02d}/{len(PRIMARY53)}] "
        f"{country}"
    )


    weight = pd.to_numeric(
        df[
            weight_col
        ],
        errors="coerce"
    )


    weight_valid = (
        np.isfinite(
            weight
        )
        &
        (
            weight
            >
            0
        )
    )


    school_valid = (
        df[
            school_col
        ]
        .notna()
    )


    for outcome in OUTCOMES:

        y = pd.to_numeric(
            df[
                outcome
            ],
            errors="coerce"
        )


        outcome_valid = np.isfinite(
            y
        )


        mask = (
            outcome_valid
            &
            weight_valid
            &
            school_valid
        )


        temp_y = y[
            mask
        ]


        temp_school = (
            df.loc[
                mask,
                school_col
            ]
        )


        n_students = int(
            mask.sum()
        )


        n_schools = int(
            temp_school.nunique()
        )


        if outcome == "BEINGBULLIED":

            outcome_kind = (
                "Regression"
            )


            n_positive = np.nan

            n_negative = np.nan

            prevalence = np.nan

            y_unique = int(
                temp_y.nunique()
            )


        else:

            outcome_kind = (
                "Classification"
            )


            n_positive = int(
                (
                    temp_y
                    ==
                    1
                )
                .sum()
            )


            n_negative = int(
                (
                    temp_y
                    ==
                    0
                )
                .sum()
            )


            prevalence = (
                n_positive
                /
                n_students
                if n_students
                >
                0
                else np.nan
            )


            y_unique = int(
                temp_y.nunique()
            )


        sample_rows.append(
            {
                "Country":
                    country,

                "Outcome":
                    outcome,

                "OutcomeKind":
                    outcome_kind,

                "N_Students":
                    n_students,

                "N_Schools":
                    n_schools,

                "OutcomeUnique":
                    y_unique,

                "N_Positive":
                    n_positive,

                "N_Negative":
                    n_negative,

                "Prevalence":
                    prevalence,

                "WeightSum":
                    float(
                        weight[
                            mask
                        ]
                        .sum()
                    )
                    if n_students
                    >
                    0
                    else np.nan,
            }
        )


    del df

    gc.collect()


sample_audit = pd.DataFrame(
    sample_rows
)


sample_audit.to_csv(
    OUT_DIR
    / "expanded_primary53_outcome_sample_audit.csv",
    index=False
)


# ============================================================
# 9. Basic fold-feasibility checks
#
# Baseline design uses 5 school-grouped folds.
# ============================================================

sample_audit[
    "FiveFoldSchoolFeasible"
] = (
    sample_audit[
        "N_Schools"
    ]
    >=
    5
)


sample_audit[
    "ClassificationHasBothClasses"
] = np.where(

    sample_audit[
        "OutcomeKind"
    ]
    ==
    "Classification",

    (
        (
            sample_audit[
                "N_Positive"
            ]
            >
            0
        )
        &
        (
            sample_audit[
                "N_Negative"
            ]
            >
            0
        )
    ),

    True
)


sample_audit[
    "BasicEligible"
] = (
    (
        sample_audit[
            "N_Students"
        ]
        >
        0
    )
    &
    sample_audit[
        "FiveFoldSchoolFeasible"
    ]
    &
    sample_audit[
        "ClassificationHasBothClasses"
    ]
)


sample_audit.to_csv(
    OUT_DIR
    / "expanded_primary53_outcome_sample_audit.csv",
    index=False
)


# ============================================================
# 10. Parent12 outcome sample audit
#
# Same definition, just mark which Primary53 outcome rows
# belong to the Full47 sensitivity cohort.
# ============================================================

parent12_audit = (
    sample_audit[
        sample_audit[
            "Country"
        ]
        .isin(
            FULL_PARENT12
        )
    ]
    .copy()
)


parent12_audit.to_csv(
    OUT_DIR
    / "expanded_parent12_outcome_sample_audit.csv",
    index=False
)


# ============================================================
# 11. Determine actual frozen model universe
# ============================================================

eligible_primary_models = (
    sample_audit[
        sample_audit[
            "BasicEligible"
        ]
    ][
        [
            "Country",
            "Outcome",
        ]
    ]
    .copy()
)


eligible_parent_models = (
    parent12_audit[
        parent12_audit[
            "BasicEligible"
        ]
    ][
        [
            "Country",
            "Outcome",
        ]
    ]
    .copy()
)


eligible_primary_models.to_csv(
    OUT_DIR
    / "frozen_primary53_country_outcome_models.csv",
    index=False
)


eligible_parent_models.to_csv(
    OUT_DIR
    / "frozen_parent12_country_outcome_models.csv",
    index=False
)


# ============================================================
# 12. Summary by outcome
# ============================================================

outcome_summary = (
    sample_audit
    .groupby(
        "Outcome",
        as_index=False
    )
    .agg(
        SystemsAudited=(
            "Country",
            "nunique"
        ),

        EligibleSystems=(
            "BasicEligible",
            "sum"
        ),

        MedianStudents=(
            "N_Students",
            "median"
        ),

        MinStudents=(
            "N_Students",
            "min"
        ),

        MedianSchools=(
            "N_Schools",
            "median"
        ),

        MinSchools=(
            "N_Schools",
            "min"
        ),
    )
)


outcome_summary.to_csv(
    OUT_DIR
    / "expanded_primary53_outcome_summary.csv",
    index=False
)


# ============================================================
# 13. Print schema QA
# ============================================================

print("\n")
print("=" * 120)
print("PRIMARY53 SCHEMA QA")
print("=" * 120)


for column in [
    "SchoolColumnPresent",
    "WeightColumnPresent",
    "StudentColumnPresent",
    "AllOutcomesPresent",
    "AllCore14Present",
    "AllExpanded43Present",
    "All30PVColumnsPresent",
]:

    if column in schema_check.columns:

        print(
            f"{column}:",
            schema_check[
                column
            ]
            .value_counts(
                dropna=False
            )
            .to_dict()
        )


# ============================================================
# 14. Print outcome summary
# ============================================================

print("\n")
print("=" * 120)
print("PRIMARY53 OUTCOME SAMPLE SUMMARY")
print("=" * 120)


print(
    outcome_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 15. Ineligible country × outcome combinations
# ============================================================

print("\n")
print("=" * 120)
print("INELIGIBLE PRIMARY53 COUNTRY × OUTCOME COMBINATIONS")
print("=" * 120)


ineligible = (
    sample_audit[
        ~sample_audit[
            "BasicEligible"
        ]
    ]
)


if len(
    ineligible
) == 0:

    print(
        "None."
    )


else:

    print(
        ineligible[
            [
                "Country",
                "Outcome",
                "N_Students",
                "N_Schools",
                "OutcomeUnique",
                "N_Positive",
                "N_Negative",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ============================================================
# 16. Classification prevalence extremes
# ============================================================

print("\n")
print("=" * 120)
print("CLASSIFICATION PREVALENCE EXTREMES")
print("=" * 120)


classification_rows = (
    sample_audit[
        sample_audit[
            "OutcomeKind"
        ]
        ==
        "Classification"
    ]
)


for outcome in [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]:

    temp = (
        classification_rows[
            classification_rows[
                "Outcome"
            ]
            ==
            outcome
        ]
        .sort_values(
            "Prevalence"
        )
    )


    print(
        "\n",
        outcome
    )


    print(
        "Lowest:"
    )


    print(
        temp[
            [
                "Country",
                "N_Students",
                "Prevalence",
            ]
        ]
        .head(
            5
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


    print(
        "Highest:"
    )


    print(
        temp[
            [
                "Country",
                "N_Students",
                "Prevalence",
            ]
        ]
        .tail(
            5
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 17. Final frozen design
# ============================================================

print("\n")
print("=" * 120)
print("PHASE 3C FROZEN DESIGN")
print("=" * 120)


print(
    "Primary system cohort:",
    len(
        PRIMARY53
    )
)


print(
    "Literal parent sensitivity cohort:",
    len(
        FULL_PARENT12
    )
)


print(
    "Eligible primary country × outcome models:",
    len(
        eligible_primary_models
    )
)


print(
    "Eligible parent country × outcome models:",
    len(
        eligible_parent_models
    )
)


print(
    "\nMATCHED MODEL COMPARISONS:"
)


print(
    "1. Core14 vs Expanded43"
)


print(
    "2. Core17 vs Expanded46"
)


print(
    "3. Expanded46 vs Full47 within Parent12"
)


print(
    "\nPredictor missingness will NOT be used to drop "
    "individual students."
)


print(
    "All matched comparisons will use identical "
    "outcome-valid rows and identical school folds."
)


print(
    "\nNo XGBoost models have been fitted yet."
)

Primary expanded cohort: 53 systems
['ARE', 'ARG', 'AUT', 'BGR', 'BRA', 'BRN', 'CHE', 'CHL', 'COL', 'CRI', 'CZE', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GEO', 'GRC', 'HKG', 'HRV', 'HUN', 'IDN', 'IRL', 'ISL', 'JOR', 'KAZ', 'KOR', 'KSV', 'LTU', 'LUX', 'LVA', 'MAC', 'MDA', 'MEX', 'MLT', 'MNE', 'NLD', 'PHL', 'POL', 'PRT', 'QAT', 'QAZ', 'QCI', 'ROU', 'SRB', 'SVK', 'SVN', 'SWE', 'TAP', 'THA', 'TUR', 'URY', 'USA']

Literal Full47 parent-module cohort: 12 systems
['BRA', 'CHL', 'GEO', 'HKG', 'HRV', 'IRL', 'KOR', 'LUX', 'MAC', 'MEX', 'MLT', 'PRT']

Resolved school ID: CNTSCHID
Resolved student ID: CNTSTUID
Resolved student weight: W_FSTUWT
[01/53] ARE
[02/53] ARG
[03/53] AUT
[04/53] BGR
[05/53] BRA
[06/53] BRN
[07/53] CHE
[08/53] CHL
[09/53] COL
[10/53] CRI
[11/53] CZE
[12/53] ESP
[13/53] EST
[14/53] FIN
[15/53] FRA
[16/53] GBR
[17/53] GEO
[18/53] GRC
[19/53] HKG
[20/53] HRV
[21/53] HUN
[22/53] IDN
[23/53] IRL
[24/53] ISL
[25/53] JOR
[26/53] KAZ
[27/53] KOR
[28/53] KSV
[29/53] LTU
[30/53] LUX
[31/

In [4]:
# ============================================================
# PHASE 3D — MATCHED CORE14 vs EXPANDED43 MODELS
# ============================================================
#
# PRIMARY EXPANDED-PREDICTOR VALIDATION
#
# QUESTION
# --------
# When the original 14 predictors compete against the broader
# 43-predictor contextual candidate set:
#
#   1. Does predictive performance improve?
#   2. Does BELONG remain dominant?
#   3. How many of the original Core14 remain in the top 5/10?
#   4. How much total attribution remains in the Core14?
#   5. Is the ordering of the Core14 preserved?
#   6. Which NEW predictors enter the top ranks?
#
#
# MATCHED DESIGN
# --------------
# For each Country × Outcome:
#
#   Core14 and Expanded43 use:
#
#       - identical students
#       - identical outcome values
#       - identical survey weights
#       - identical school-grouped folds
#
# Predictor missingness DOES NOT remove students.
# XGBoost handles NaN natively.
#
#
# FROZEN MODEL SPECIFICATION
# --------------------------
#
#   300 trees
#   max_depth = 4
#   learning_rate = .05
#   subsample = .8
#   colsample_bytree = .8
#   min_child_weight = 5
#   lambda = 4
#   alpha = .6
#   gamma = 0
#   hist
#
# Regression:
#   GroupKFold by school
#
# Classification:
#   StratifiedGroupKFold by school
#   shuffle = True
#   random_state = 42
#
# SHAP:
#   exact native XGBoost TreeSHAP
#   held-out students only
#
# ============================================================


from pathlib import Path
import pickle
import gc
import time

import numpy as np
import pandas as pd

import xgboost as xgb

from xgboost import (
    XGBRegressor,
    XGBClassifier,
)

from sklearn.model_selection import (
    GroupKFold,
    StratifiedGroupKFold,
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    roc_auc_score,
    average_precision_score,
)

from scipy.stats import spearmanr


# ============================================================
# 1. Paths
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "expanded_predictor_analysis"
)


CHECKPOINT_DIR = (
    OUT_DIR
    / "core14_vs_expanded43_checkpoints"
)


FOLD_DIR = (
    OUT_DIR
    / "frozen_primary53_folds"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FOLD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MODEL_LIST_PATH = (
    OUT_DIR
    / "frozen_primary53_country_outcome_models.csv"
)


# ============================================================
# 2. Frozen columns
# ============================================================

SCHOOL_COL = "CNTSCHID"

STUDENT_COL = "CNTSTUID"

WEIGHT_COL = "W_FSTUWT"


# ============================================================
# 3. Predictor sets
# ============================================================

CORE14_MODEL = list(
    CORE14
)


EXPANDED43_MODEL = [
    predictor
    for predictor
    in EXPANDED_NONACHIEVEMENT
    if predictor
    !=
    "PA006Q09TA"
]


assert len(
    CORE14_MODEL
) == 14


assert len(
    EXPANDED43_MODEL
) == 43


assert set(
    CORE14_MODEL
).issubset(
    set(
        EXPANDED43_MODEL
    )
)


NEW29 = [
    predictor
    for predictor
    in EXPANDED43_MODEL
    if predictor
    not in CORE14_MODEL
]


assert len(
    NEW29
) == 29


# ============================================================
# 4. Frozen XGBoost parameters
# ============================================================

XGB_COMMON = {

    "n_estimators":
        300,

    "max_depth":
        4,

    "learning_rate":
        0.05,

    "subsample":
        0.8,

    "colsample_bytree":
        0.8,

    "min_child_weight":
        5,

    "reg_lambda":
        4.0,

    "reg_alpha":
        0.6,

    "gamma":
        0.0,

    "random_state":
        42,

    "n_jobs":
        -1,

    "tree_method":
        "hist",
}


# ============================================================
# 5. Load frozen country × outcome universe
# ============================================================

model_universe = pd.read_csv(
    MODEL_LIST_PATH
)


model_universe = (
    model_universe
    .sort_values(
        [
            "Country",
            "Outcome"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Frozen country × outcome models:",
    len(
        model_universe
    )
)


print(
    "\nBy outcome:"
)


print(
    model_universe[
        "Outcome"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 6. Numeric predictor conversion
# ============================================================

def numeric_frame(
    df,
    features
):

    output = pd.DataFrame(
        index=df.index
    )


    for feature in features:

        output[
            feature
        ] = pd.to_numeric(
            df[
                feature
            ],
            errors="coerce"
        )


    return output


# ============================================================
# 7. Build exact frozen fold assignments
# ============================================================

def build_fold_ids(
    y,
    groups,
    outcome
):

    y = np.asarray(
        y
    )


    groups = np.asarray(
        groups
    )


    n = len(
        y
    )


    fold_ids = np.full(
        n,
        -1,
        dtype=int
    )


    dummy_X = np.zeros(
        (
            n,
            1
        ),
        dtype=float
    )


    if outcome == "BEINGBULLIED":

        splitter = GroupKFold(
            n_splits=5
        )


        split_iterator = splitter.split(
            dummy_X,
            y,
            groups=groups
        )


    else:

        splitter = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )


        split_iterator = splitter.split(
            dummy_X,
            y,
            groups=groups
        )


    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        split_iterator
    ):

        fold_ids[
            test_idx
        ] = fold


    if (
        fold_ids
        <
        0
    ).any():

        raise RuntimeError(
            "Some observations were not assigned "
            "to a fold."
        )


    return fold_ids


# ============================================================
# 8. Model constructor
# ============================================================

def make_model(
    outcome
):

    if outcome == "BEINGBULLIED":

        return XGBRegressor(

            objective=
                "reg:squarederror",

            **XGB_COMMON
        )


    return XGBClassifier(

        objective=
            "binary:logistic",

        eval_metric=
            "logloss",

        **XGB_COMMON
    )


# ============================================================
# 9. Prediction helper
# ============================================================

def predict_model(
    model,
    X,
    outcome
):

    if outcome == "BEINGBULLIED":

        return model.predict(
            X
        )


    return model.predict_proba(
        X
    )[
        :,
        1
    ]


# ============================================================
# 10. Weighted performance
# ============================================================

def calculate_performance(
    y,
    pred,
    weights,
    outcome
):

    y = np.asarray(
        y,
        dtype=float
    )


    pred = np.asarray(
        pred,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    if outcome == "BEINGBULLIED":

        return {

            "R2":
                float(
                    r2_score(
                        y,
                        pred,
                        sample_weight=weights
                    )
                ),

            "MAE":
                float(
                    mean_absolute_error(
                        y,
                        pred,
                        sample_weight=weights
                    )
                ),

            "ROC_AUC":
                np.nan,

            "PR_AUC":
                np.nan,
        }


    return {

        "R2":
            np.nan,

        "MAE":
            np.nan,

        "ROC_AUC":
            float(
                roc_auc_score(
                    y,
                    pred,
                    sample_weight=weights
                )
            ),

        "PR_AUC":
            float(
                average_precision_score(
                    y,
                    pred,
                    sample_weight=weights
                )
            ),
    }


# ============================================================
# 11. Native held-out TreeSHAP
# ============================================================

def native_shap_values(
    model,
    X,
    features
):

    dmatrix = xgb.DMatrix(
        X,
        feature_names=features,
        missing=np.nan
    )


    contributions = (
        model
        .get_booster()
        .predict(
            dmatrix,
            pred_contribs=True,
            approx_contribs=False
        )
    )


    # Final column = expected-value bias term
    shap_values = (
        contributions[
            :,
            :-1
        ]
    )


    if (
        shap_values.shape[
            1
        ]
        !=
        len(
            features
        )
    ):

        raise RuntimeError(
            "Unexpected SHAP matrix width."
        )


    return shap_values


# ============================================================
# 12. Fit one predictor set with frozen folds
# ============================================================

def fit_oof_model(
    X,
    y,
    weights,
    fold_ids,
    outcome,
    features
):

    n = len(
        y
    )


    oof_predictions = np.full(
        n,
        np.nan,
        dtype=float
    )


    weighted_abs_shap_sum = np.zeros(
        len(
            features
        ),
        dtype=float
    )


    shap_weight_sum = 0.0


    fold_metric_rows = []


    for fold in range(
        5
    ):

        train_mask = (
            fold_ids
            !=
            fold
        )


        test_mask = (
            fold_ids
            ==
            fold
        )


        X_train = (
            X.loc[
                train_mask,
                features
            ]
        )


        X_test = (
            X.loc[
                test_mask,
                features
            ]
        )


        y_train = (
            y[
                train_mask
            ]
        )


        y_test = (
            y[
                test_mask
            ]
        )


        w_train = (
            weights[
                train_mask
            ]
        )


        w_test = (
            weights[
                test_mask
            ]
        )


        # ----------------------------------------------------
        # Normalize training survey weights to mean 1.
        #
        # Same frozen convention used in baseline models.
        # ----------------------------------------------------

        w_train_normalized = (
            w_train
            /
            np.mean(
                w_train
            )
        )


        # ----------------------------------------------------
        # Fold class QA
        # ----------------------------------------------------

        if outcome != "BEINGBULLIED":

            if len(
                np.unique(
                    y_train
                )
            ) != 2:

                raise RuntimeError(
                    "Training fold has only one class."
                )


            if len(
                np.unique(
                    y_test
                )
            ) != 2:

                raise RuntimeError(
                    "Test fold has only one class."
                )


        # ----------------------------------------------------
        # Fit
        # ----------------------------------------------------

        model = make_model(
            outcome
        )


        model.fit(
            X_train,
            y_train,
            sample_weight=
                w_train_normalized
        )


        # ----------------------------------------------------
        # Held-out prediction
        # ----------------------------------------------------

        pred = predict_model(
            model,
            X_test,
            outcome
        )


        oof_predictions[
            test_mask
        ] = pred


        # ----------------------------------------------------
        # Held-out exact TreeSHAP
        # ----------------------------------------------------

        shap_values = native_shap_values(
            model,
            X_test,
            features
        )


        weighted_abs_shap_sum += (
            np.sum(
                np.abs(
                    shap_values
                )
                *
                w_test[
                    :,
                    None
                ],
                axis=0
            )
        )


        shap_weight_sum += float(
            np.sum(
                w_test
            )
        )


        # ----------------------------------------------------
        # Fold-level performance
        # ----------------------------------------------------

        fold_perf = calculate_performance(
            y_test,
            pred,
            w_test,
            outcome
        )


        fold_metric_rows.append(
            {
                "Fold":
                    fold,

                "N_Train":
                    int(
                        train_mask.sum()
                    ),

                "N_Test":
                    int(
                        test_mask.sum()
                    ),

                **fold_perf,
            }
        )


        del (
            model,
            X_train,
            X_test,
            shap_values
        )


        gc.collect()


    # ========================================================
    # OOF integrity
    # ========================================================

    if np.isnan(
        oof_predictions
    ).any():

        raise RuntimeError(
            "Missing OOF predictions."
        )


    # ========================================================
    # Overall weighted OOF performance
    # ========================================================

    performance = calculate_performance(
        y,
        oof_predictions,
        weights,
        outcome
    )


    # ========================================================
    # Weighted mean absolute OOF TreeSHAP
    # ========================================================

    mean_abs_shap = (
        weighted_abs_shap_sum
        /
        shap_weight_sum
    )


    total = np.sum(
        mean_abs_shap
    )


    if (
        not np.isfinite(
            total
        )
        or
        total
        <=
        0
    ):

        raise RuntimeError(
            "Invalid total SHAP attribution."
        )


    importance_share = (
        mean_abs_shap
        /
        total
    )


    attribution = pd.DataFrame(
        {
            "Feature":
                features,

            "MeanAbsSHAP":
                mean_abs_shap,

            "ImportanceShare":
                importance_share,
        }
    )


    attribution = (
        attribution
        .sort_values(
            "ImportanceShare",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    attribution[
        "Rank"
    ] = (
        np.arange(
            1,
            len(
                attribution
            )
            +
            1
        )
    )


    return {

        "performance":
            performance,

        "attribution":
            attribution,

        "fold_metrics":
            pd.DataFrame(
                fold_metric_rows
            ),

        "oof_predictions":
            oof_predictions,
    }


# ============================================================
# 13. Cosine similarity helper
# ============================================================

def cosine_similarity(
    a,
    b
):

    a = np.asarray(
        a,
        dtype=float
    )


    b = np.asarray(
        b,
        dtype=float
    )


    denominator = (
        np.linalg.norm(
            a
        )
        *
        np.linalg.norm(
            b
        )
    )


    if denominator == 0:

        return np.nan


    return float(
        np.dot(
            a,
            b
        )
        /
        denominator
    )


# ============================================================
# 14. Core-preservation comparison
# ============================================================

def compare_architectures(
    core_attr,
    expanded_attr
):

    core = (
        core_attr
        .set_index(
            "Feature"
        )
        .loc[
            CORE14_MODEL
        ]
        .copy()
    )


    expanded = (
        expanded_attr
        .set_index(
            "Feature"
        )
        .copy()
    )


    expanded_core = (
        expanded
        .loc[
            CORE14_MODEL
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Total Expanded43 attribution allocated to original Core14
    # --------------------------------------------------------

    core_mass = float(
        expanded_core[
            "ImportanceShare"
        ]
        .sum()
    )


    # --------------------------------------------------------
    # Renormalize shared 14 inside Expanded43
    # --------------------------------------------------------

    expanded_core_renorm = (
        expanded_core[
            "ImportanceShare"
        ]
        /
        core_mass
    )


    baseline_core_share = (
        core[
            "ImportanceShare"
        ]
    )


    # --------------------------------------------------------
    # Rank order among only the shared Core14 predictors
    # --------------------------------------------------------

    core_rank_shared = (
        baseline_core_share
        .rank(
            ascending=False,
            method="min"
        )
    )


    expanded_rank_shared = (
        expanded_core_renorm
        .rank(
            ascending=False,
            method="min"
        )
    )


    rank_rho = spearmanr(
        core_rank_shared,
        expanded_rank_shared
    ).statistic


    cosine = cosine_similarity(
        baseline_core_share,
        expanded_core_renorm
    )


    # --------------------------------------------------------
    # Expanded global ranks
    # --------------------------------------------------------

    expanded_global_ranks = (
        expanded[
            "Rank"
        ]
    )


    n_core_top5 = int(
        (
            expanded_global_ranks
            .loc[
                CORE14_MODEL
            ]
            <=
            5
        )
        .sum()
    )


    n_core_top10 = int(
        (
            expanded_global_ranks
            .loc[
                CORE14_MODEL
            ]
            <=
            10
        )
        .sum()
    )


    belong_rank = int(
        expanded.loc[
            "BELONG",
            "Rank"
        ]
    )


    belong_share = float(
        expanded.loc[
            "BELONG",
            "ImportanceShare"
        ]
    )


    # --------------------------------------------------------
    # New features entering top 10
    # --------------------------------------------------------

    top10 = (
        expanded[
            expanded[
                "Rank"
            ]
            <=
            10
        ]
    )


    new_top10 = (
        top10[
            top10.index.isin(
                NEW29
            )
        ]
        .sort_values(
            "Rank"
        )
    )


    new_top10_features = "|".join(
        new_top10.index.tolist()
    )


    return {

        "Core14AttributionMass":
            core_mass,

        "Core14RenormCosine":
            cosine,

        "Core14SharedRankSpearman":
            float(
                rank_rho
            ),

        "N_Core14_in_Top5":
            n_core_top5,

        "N_Core14_in_Top10":
            n_core_top10,

        "BELONG_ExpandedRank":
            belong_rank,

        "BELONG_ExpandedShare":
            belong_share,

        "N_New29_in_Top10":
            int(
                len(
                    new_top10
                )
            ),

        "NewTop10Features":
            new_top10_features,
    }


# ============================================================
# 15. Main fitting loop
# ============================================================

total_pairs = len(
    model_universe
)


start_time = time.time()


for pair_index, row in (
    model_universe
    .iterrows()
):

    country = row[
        "Country"
    ]


    outcome = row[
        "Outcome"
    ]


    checkpoint_path = (
        CHECKPOINT_DIR
        /
        f"{country}__{outcome}.pkl"
    )


    fold_path = (
        FOLD_DIR
        /
        f"{country}__{outcome}__folds.pkl"
    )


    # --------------------------------------------------------
    # Resume support
    # --------------------------------------------------------

    if checkpoint_path.exists():

        print(
            f"[{pair_index + 1:03d}/{total_pairs}] "
            f"{country} {outcome} — already complete"
        )


        continue


    pair_start = time.time()


    print(
        f"\n[{pair_index + 1:03d}/{total_pairs}] "
        f"{country} | {outcome}"
    )


    # --------------------------------------------------------
    # Load processed data
    # --------------------------------------------------------

    path = (
        PROCESSED_DIR
        /
        f"{country}_analysis.pkl"
    )


    df = pd.read_pickle(
        path
    )


    # --------------------------------------------------------
    # Identical outcome-valid sample
    # --------------------------------------------------------

    y_full = pd.to_numeric(
        df[
            outcome
        ],
        errors="coerce"
    )


    weight_full = pd.to_numeric(
        df[
            WEIGHT_COL
        ],
        errors="coerce"
    )


    valid_mask = (

        np.isfinite(
            y_full
        )

        &

        np.isfinite(
            weight_full
        )

        &

        (
            weight_full
            >
            0
        )

        &

        df[
            SCHOOL_COL
        ]
        .notna()
    )


    selected = (
        df.loc[
            valid_mask
        ]
        .copy()
    )


    y = (
        pd.to_numeric(
            selected[
                outcome
            ],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    weights = (
        pd.to_numeric(
            selected[
                WEIGHT_COL
            ],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    groups = (
        selected[
            SCHOOL_COL
        ]
        .to_numpy()
    )


    student_ids = (
        selected[
            STUDENT_COL
        ]
        .astype(
            str
        )
        .to_numpy()
    )


    # --------------------------------------------------------
    # Student-ID QA
    # --------------------------------------------------------

    if len(
        np.unique(
            student_ids
        )
    ) != len(
        student_ids
    ):

        raise RuntimeError(
            f"Duplicate student IDs in "
            f"{country} {outcome}."
        )


    # --------------------------------------------------------
    # Build predictor matrix once
    # --------------------------------------------------------

    X = numeric_frame(
        selected,
        EXPANDED43_MODEL
    )


    # --------------------------------------------------------
    # Exact shared folds
    # --------------------------------------------------------

    fold_ids = build_fold_ids(
        y,
        groups,
        outcome
    )


    # --------------------------------------------------------
    # Save frozen fold assignment for later PV analysis
    # --------------------------------------------------------

    fold_table = pd.DataFrame(
        {
            "Country":
                country,

            "Outcome":
                outcome,

            "CNTSTUID":
                student_ids,

            "Fold":
                fold_ids,
        }
    )


    fold_table.to_pickle(
        fold_path
    )


    # --------------------------------------------------------
    # Fold QA
    # --------------------------------------------------------

    print(
        "   N students:",
        len(
            y
        ),
        "| schools:",
        pd.Series(
            groups
        ).nunique()
    )


    print(
        "   Fold sizes:",
        pd.Series(
            fold_ids
        )
        .value_counts()
        .sort_index()
        .to_dict()
    )


    # ========================================================
    # CORE14
    # ========================================================

    print(
        "   fitting Core14..."
    )


    core_result = fit_oof_model(

        X=X,

        y=y,

        weights=weights,

        fold_ids=fold_ids,

        outcome=outcome,

        features=CORE14_MODEL,
    )


    # ========================================================
    # EXPANDED43
    # ========================================================

    print(
        "   fitting Expanded43..."
    )


    expanded_result = fit_oof_model(

        X=X,

        y=y,

        weights=weights,

        fold_ids=fold_ids,

        outcome=outcome,

        features=EXPANDED43_MODEL,
    )


    # ========================================================
    # Matched comparison
    # ========================================================

    architecture_comparison = (
        compare_architectures(
            core_result[
                "attribution"
            ],
            expanded_result[
                "attribution"
            ]
        )
    )


    # --------------------------------------------------------
    # Performance deltas
    # --------------------------------------------------------

    if outcome == "BEINGBULLIED":

        performance_delta = {

            "PrimaryMetric":
                "R2",

            "CorePrimaryMetric":
                core_result[
                    "performance"
                ][
                    "R2"
                ],

            "ExpandedPrimaryMetric":
                expanded_result[
                    "performance"
                ][
                    "R2"
                ],

            "DeltaPrimaryMetric":
                (
                    expanded_result[
                        "performance"
                    ][
                        "R2"
                    ]
                    -
                    core_result[
                        "performance"
                    ][
                        "R2"
                    ]
                ),
        }


    else:

        performance_delta = {

            "PrimaryMetric":
                "ROC_AUC",

            "CorePrimaryMetric":
                core_result[
                    "performance"
                ][
                    "ROC_AUC"
                ],

            "ExpandedPrimaryMetric":
                expanded_result[
                    "performance"
                ][
                    "ROC_AUC"
                ],

            "DeltaPrimaryMetric":
                (
                    expanded_result[
                        "performance"
                    ][
                        "ROC_AUC"
                    ]
                    -
                    core_result[
                        "performance"
                    ][
                        "ROC_AUC"
                    ]
                ),
        }


    # ========================================================
    # Add identifiers to attribution tables
    # ========================================================

    core_attr = (
        core_result[
            "attribution"
        ]
        .copy()
    )


    core_attr.insert(
        0,
        "Model",
        "Core14"
    )


    core_attr.insert(
        0,
        "Outcome",
        outcome
    )


    core_attr.insert(
        0,
        "Country",
        country
    )


    expanded_attr = (
        expanded_result[
            "attribution"
        ]
        .copy()
    )


    expanded_attr.insert(
        0,
        "Model",
        "Expanded43"
    )


    expanded_attr.insert(
        0,
        "Outcome",
        outcome
    )


    expanded_attr.insert(
        0,
        "Country",
        country
    )


    # ========================================================
    # Checkpoint object
    # ========================================================

    checkpoint = {

        "Country":
            country,

        "Outcome":
            outcome,

        "N_Students":
            len(
                y
            ),

        "N_Schools":
            int(
                pd.Series(
                    groups
                )
                .nunique()
            ),

        "CorePerformance":
            core_result[
                "performance"
            ],

        "ExpandedPerformance":
            expanded_result[
                "performance"
            ],

        "PerformanceComparison":
            performance_delta,

        "ArchitectureComparison":
            architecture_comparison,

        "CoreAttribution":
            core_attr,

        "ExpandedAttribution":
            expanded_attr,

        "CoreFoldMetrics":
            core_result[
                "fold_metrics"
            ],

        "ExpandedFoldMetrics":
            expanded_result[
                "fold_metrics"
            ],
    }


    with open(
        checkpoint_path,
        "wb"
    ) as file:

        pickle.dump(
            checkpoint,
            file
        )


    elapsed = (
        time.time()
        -
        pair_start
    )


    print(
        f"   complete in {elapsed:.1f}s"
    )


    print(
        "   primary metric:",
        performance_delta[
            "PrimaryMetric"
        ],
        "| Core =",
        round(
            performance_delta[
                "CorePrimaryMetric"
            ],
            4
        ),
        "| Expanded =",
        round(
            performance_delta[
                "ExpandedPrimaryMetric"
            ],
            4
        ),
        "| Δ =",
        round(
            performance_delta[
                "DeltaPrimaryMetric"
            ],
            4
        )
    )


    print(
        "   Core attribution mass =",
        round(
            architecture_comparison[
                "Core14AttributionMass"
            ],
            4
        ),
        "| BELONG rank =",
        architecture_comparison[
            "BELONG_ExpandedRank"
        ],
        "| Core in top10 =",
        architecture_comparison[
            "N_Core14_in_Top10"
        ]
    )


    del (
        df,
        selected,
        X,
        core_result,
        expanded_result,
        checkpoint
    )


    gc.collect()


# ============================================================
# 16. Reassemble ALL checkpoints
# ============================================================

performance_rows = []

comparison_rows = []

all_attribution = []


for _, row in (
    model_universe.iterrows()
):

    country = row[
        "Country"
    ]


    outcome = row[
        "Outcome"
    ]


    checkpoint_path = (
        CHECKPOINT_DIR
        /
        f"{country}__{outcome}.pkl"
    )


    if not checkpoint_path.exists():

        raise RuntimeError(
            f"Missing checkpoint: "
            f"{country} {outcome}"
        )


    with open(
        checkpoint_path,
        "rb"
    ) as file:

        result = pickle.load(
            file
        )


    core_perf = (
        result[
            "CorePerformance"
        ]
    )


    expanded_perf = (
        result[
            "ExpandedPerformance"
        ]
    )


    # --------------------------------------------------------
    # Core performance
    # --------------------------------------------------------

    performance_rows.append(
        {
            "Country":
                country,

            "Outcome":
                outcome,

            "Model":
                "Core14",

            "N_Students":
                result[
                    "N_Students"
                ],

            "N_Schools":
                result[
                    "N_Schools"
                ],

            **core_perf,
        }
    )


    # --------------------------------------------------------
    # Expanded performance
    # --------------------------------------------------------

    performance_rows.append(
        {
            "Country":
                country,

            "Outcome":
                outcome,

            "Model":
                "Expanded43",

            "N_Students":
                result[
                    "N_Students"
                ],

            "N_Schools":
                result[
                    "N_Schools"
                ],

            **expanded_perf,
        }
    )


    # --------------------------------------------------------
    # Matched comparison
    # --------------------------------------------------------

    comparison_rows.append(
        {
            "Country":
                country,

            "Outcome":
                outcome,

            "N_Students":
                result[
                    "N_Students"
                ],

            "N_Schools":
                result[
                    "N_Schools"
                ],

            **result[
                "PerformanceComparison"
            ],

            **result[
                "ArchitectureComparison"
            ],
        }
    )


    all_attribution.append(
        result[
            "CoreAttribution"
        ]
    )


    all_attribution.append(
        result[
            "ExpandedAttribution"
        ]
    )


performance_df = pd.DataFrame(
    performance_rows
)


comparison_df = pd.DataFrame(
    comparison_rows
)


attribution_df = pd.concat(
    all_attribution,
    ignore_index=True
)


# ============================================================
# 17. Save complete results
# ============================================================

performance_df.to_csv(
    OUT_DIR
    / "core14_vs_expanded43_performance.csv",
    index=False
)


comparison_df.to_csv(
    OUT_DIR
    / "core14_vs_expanded43_matched_comparison.csv",
    index=False
)


attribution_df.to_csv(
    OUT_DIR
    / "core14_vs_expanded43_oof_shap_rankings.csv",
    index=False
)


# ============================================================
# 18. Performance summary by outcome
# ============================================================

performance_summary_rows = []


for outcome in sorted(
    comparison_df[
        "Outcome"
    ]
    .unique()
):

    temp = (
        comparison_df[
            comparison_df[
                "Outcome"
            ]
            ==
            outcome
        ]
    )


    performance_summary_rows.append(
        {
            "Outcome":
                outcome,

            "N_Systems":
                len(
                    temp
                ),

            "PrimaryMetric":
                temp[
                    "PrimaryMetric"
                ]
                .iloc[
                    0
                ],

            "Mean_Core":
                temp[
                    "CorePrimaryMetric"
                ]
                .mean(),

            "Mean_Expanded":
                temp[
                    "ExpandedPrimaryMetric"
                ]
                .mean(),

            "Mean_Delta":
                temp[
                    "DeltaPrimaryMetric"
                ]
                .mean(),

            "Median_Delta":
                temp[
                    "DeltaPrimaryMetric"
                ]
                .median(),

            "Min_Delta":
                temp[
                    "DeltaPrimaryMetric"
                ]
                .min(),

            "Max_Delta":
                temp[
                    "DeltaPrimaryMetric"
                ]
                .max(),

            "N_ExpandedBetter":
                int(
                    (
                        temp[
                            "DeltaPrimaryMetric"
                        ]
                        >
                        0
                    )
                    .sum()
                ),

            "Pct_ExpandedBetter":
                float(
                    (
                        temp[
                            "DeltaPrimaryMetric"
                        ]
                        >
                        0
                    )
                    .mean()
                    *
                    100
                ),
        }
    )


performance_summary = pd.DataFrame(
    performance_summary_rows
)


performance_summary.to_csv(
    OUT_DIR
    / "core14_vs_expanded43_performance_summary.csv",
    index=False
)


# ============================================================
# 19. Core preservation summary
# ============================================================

preservation_summary = (
    comparison_df
    .groupby(
        "Outcome",
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        Mean_Core14_AttributionMass=(
            "Core14AttributionMass",
            "mean"
        ),

        Median_Core14_AttributionMass=(
            "Core14AttributionMass",
            "median"
        ),

        Mean_Core14_RenormCosine=(
            "Core14RenormCosine",
            "mean"
        ),

        Mean_Core14_RankSpearman=(
            "Core14SharedRankSpearman",
            "mean"
        ),

        Mean_Core_in_Top5=(
            "N_Core14_in_Top5",
            "mean"
        ),

        Mean_Core_in_Top10=(
            "N_Core14_in_Top10",
            "mean"
        ),

        BELONG_MeanRank=(
            "BELONG_ExpandedRank",
            "mean"
        ),

        BELONG_MedianRank=(
            "BELONG_ExpandedRank",
            "median"
        ),

        BELONG_Rank1_Count=(
            "BELONG_ExpandedRank",
            lambda x:
                int(
                    (
                        x
                        ==
                        1
                    )
                    .sum()
                )
        ),

        Mean_NewFeatures_in_Top10=(
            "N_New29_in_Top10",
            "mean"
        ),
    )
)


preservation_summary.to_csv(
    OUT_DIR
    / "core14_preservation_summary_expanded43.csv",
    index=False
)


# ============================================================
# 20. Expanded43 global attribution
#
# Equal weight to each education system.
# ============================================================

expanded_attr = (
    attribution_df[
        attribution_df[
            "Model"
        ]
        ==
        "Expanded43"
    ]
)


global_expanded = (
    expanded_attr
    .groupby(
        [
            "Outcome",
            "Feature"
        ],
        as_index=False
    )
    .agg(

        MeanShare=(
            "ImportanceShare",
            "mean"
        ),

        MedianShare=(
            "ImportanceShare",
            "median"
        ),

        MeanRank=(
            "Rank",
            "mean"
        ),

        MedianRank=(
            "Rank",
            "median"
        ),

        Rank1Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        ==
                        1
                    )
                    .sum()
                )
        ),

        Top5Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        <=
                        5
                    )
                    .sum()
                )
        ),

        Top10Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        <=
                        10
                    )
                    .sum()
                )
        ),
    )
)


global_expanded[
    "OriginalCore14"
] = (
    global_expanded[
        "Feature"
    ]
    .isin(
        CORE14_MODEL
    )
)


global_expanded = (
    global_expanded
    .sort_values(
        [
            "Outcome",
            "MeanShare"
        ],
        ascending=[
            True,
            False
        ]
    )
)


global_expanded[
    "GlobalRank"
] = (
    global_expanded
    .groupby(
        "Outcome"
    )
    .cumcount()
    +
    1
)


global_expanded.to_csv(
    OUT_DIR
    / "expanded43_global_attribution_summary.csv",
    index=False
)


# ============================================================
# 21. Core feature survival statistics
# ============================================================

core_survival = (
    expanded_attr[
        expanded_attr[
            "Feature"
        ]
        .isin(
            CORE14_MODEL
        )
    ]
    .groupby(
        [
            "Outcome",
            "Feature"
        ],
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        MeanExpandedShare=(
            "ImportanceShare",
            "mean"
        ),

        MeanExpandedRank=(
            "Rank",
            "mean"
        ),

        MedianExpandedRank=(
            "Rank",
            "median"
        ),

        Rank1Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        ==
                        1
                    )
                    .sum()
                )
        ),

        Top5Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        <=
                        5
                    )
                    .sum()
                )
        ),

        Top10Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        <=
                        10
                    )
                    .sum()
                )
        ),
    )
)


core_survival.to_csv(
    OUT_DIR
    / "expanded43_core14_survival.csv",
    index=False
)


# ============================================================
# 22. New-feature entrants
# ============================================================

new_feature_summary = (
    expanded_attr[
        expanded_attr[
            "Feature"
        ]
        .isin(
            NEW29
        )
    ]
    .groupby(
        [
            "Outcome",
            "Feature"
        ],
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        MeanShare=(
            "ImportanceShare",
            "mean"
        ),

        MeanRank=(
            "Rank",
            "mean"
        ),

        MedianRank=(
            "Rank",
            "median"
        ),

        Rank1Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        ==
                        1
                    )
                    .sum()
                )
        ),

        Top5Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        <=
                        5
                    )
                    .sum()
                )
        ),

        Top10Count=(
            "Rank",
            lambda x:
                int(
                    (
                        x
                        <=
                        10
                    )
                    .sum()
                )
        ),
    )
)


new_feature_summary = (
    new_feature_summary
    .sort_values(
        [
            "Outcome",
            "Top10Count",
            "MeanShare"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)


new_feature_summary.to_csv(
    OUT_DIR
    / "expanded43_new_feature_entrants.csv",
    index=False
)


# ============================================================
# 23. QA
# ============================================================

expected_performance_rows = (
    len(
        model_universe
    )
    *
    2
)


if len(
    performance_df
) != expected_performance_rows:

    raise RuntimeError(
        "Unexpected number of performance rows."
    )


if len(
    comparison_df
) != len(
    model_universe
):

    raise RuntimeError(
        "Unexpected number of comparison rows."
    )


share_qa = (
    attribution_df
    .groupby(
        [
            "Country",
            "Outcome",
            "Model"
        ]
    )[
        "ImportanceShare"
    ]
    .sum()
)


max_share_error = float(
    np.max(
        np.abs(
            share_qa
            -
            1.0
        )
    )
)


print(
    "\nMaximum SHAP share-sum error:",
    max_share_error
)


# ============================================================
# 24. Print performance results
# ============================================================

print("\n")
print("=" * 125)
print("CORE14 vs EXPANDED43 — PERFORMANCE")
print("=" * 125)


print(
    performance_summary
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 25. Print architecture preservation
# ============================================================

print("\n")
print("=" * 125)
print("CORE14 ARCHITECTURE PRESERVATION INSIDE EXPANDED43")
print("=" * 125)


print(
    preservation_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 26. Print top Expanded43 features
# ============================================================

print("\n")
print("=" * 125)
print("TOP EXPANDED43 FEATURES BY OUTCOME")
print("=" * 125)


for outcome in sorted(
    global_expanded[
        "Outcome"
    ]
    .unique()
):

    print(
        "\n",
        outcome
    )


    print(
        global_expanded[
            global_expanded[
                "Outcome"
            ]
            ==
            outcome
        ][
            [
                "GlobalRank",
                "Feature",
                "OriginalCore14",
                "MeanShare",
                "MeanRank",
                "Rank1Count",
                "Top5Count",
                "Top10Count",
            ]
        ]
        .head(
            15
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 27. Print strongest NEW predictors
# ============================================================

print("\n")
print("=" * 125)
print("STRONGEST NEW29 ENTRANTS")
print("=" * 125)


for outcome in sorted(
    new_feature_summary[
        "Outcome"
    ]
    .unique()
):

    print(
        "\n",
        outcome
    )


    print(
        new_feature_summary[
            new_feature_summary[
                "Outcome"
            ]
            ==
            outcome
        ][
            [
                "Feature",
                "MeanShare",
                "MeanRank",
                "Rank1Count",
                "Top5Count",
                "Top10Count",
            ]
        ]
        .head(
            12
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 28. BELONG survival
# ============================================================

print("\n")
print("=" * 125)
print("BELONG SURVIVAL IN EXPANDED43")
print("=" * 125)


belong_survival = (
    core_survival[
        core_survival[
            "Feature"
        ]
        ==
        "BELONG"
    ]
)


print(
    belong_survival
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 29. Core top-10 survival
# ============================================================

print("\n")
print("=" * 125)
print("CORE14 TOP-10 SURVIVAL INSIDE EXPANDED43")
print("=" * 125)


print(
    core_survival
    .sort_values(
        [
            "Outcome",
            "Top10Count",
            "MeanExpandedRank"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 30. Final status
# ============================================================

runtime_minutes = (
    time.time()
    -
    start_time
) / 60


print("\n")
print("=" * 125)
print("PHASE 3D CORE14 vs EXPANDED43 COMPLETE")
print("=" * 125)


print(
    "Country × outcome comparisons:",
    len(
        comparison_df
    )
)


print(
    "Performance rows:",
    len(
        performance_df
    )
)


print(
    "Runtime this invocation:",
    round(
        runtime_minutes,
        2
    ),
    "minutes"
)


print(
    "\nImportant:"
)


print(
    "Do not interpret lower Core14 SHAP shares as "
    "failure by themselves."
)


print(
    "Correlated new predictors can split attribution that was "
    "previously concentrated in one Core14 predictor."
)


print(
    "\nNext decision:"
)


print(
    "Inspect performance gain, Core14 attribution mass, "
    "BELONG survival, and new top-rank entrants before "
    "running achievement-controlled Expanded46."
)

Frozen country × outcome models: 264

By outcome:
Outcome
BEINGBULLIED        52
BULLY_PHYSICAL      53
BULLY_RELATIONAL    53
BULLY_THREAT        53
BULLY_VERBAL        53

[001/264] ARE | BEINGBULLIED
   N students: 16856 | schools: 750
   Fold sizes: {0: 3372, 1: 3371, 2: 3371, 3: 3371, 4: 3371}
   fitting Core14...
   fitting Expanded43...
   complete in 30.4s
   primary metric: R2 | Core = 0.2101 | Expanded = 0.2705 | Δ = 0.0604
   Core attribution mass = 0.5145 | BELONG rank = 1 | Core in top10 = 3

[002/264] ARE | BULLY_PHYSICAL
   N students: 17120 | schools: 752
   Fold sizes: {0: 3364, 1: 3260, 2: 3567, 3: 3534, 4: 3395}
   fitting Core14...
   fitting Expanded43...
   complete in 30.0s
   primary metric: ROC_AUC | Core = 0.7837 | Expanded = 0.8161 | Δ = 0.0324
   Core attribution mass = 0.471 | BELONG rank = 1 | Core in top10 = 3

[003/264] ARE | BULLY_RELATIONAL
   N students: 17132 | schools: 752
   Fold sizes: {0: 3367, 1: 3485, 2: 3191, 3: 3472, 4: 3617}
   fitting Core1

In [5]:
# ============================================================
# PHASE 3E0 — INSPECT FROZEN ACHIEVEMENT OUTPUTS
#
# PURPOSE:
# Reuse the already-completed Core17 plausible-value models
# instead of unnecessarily refitting them.
# ============================================================

from pathlib import Path
import pandas as pd


ACH_DIR = Path(
    "../results/achievement_pv"
)


FILES = [
    "pv_model_performance.csv",
    "pv_model_rankings.csv",
    "pv_model_qa.csv",
    "system_conceptual_predictor_summary.csv",
    "primary_vs_augmented_architecture_summary.csv",
]


for filename in FILES:

    path = (
        ACH_DIR
        /
        filename
    )

    print("\n")
    print("=" * 120)
    print(filename)
    print("=" * 120)

    print(
        "Exists:",
        path.exists()
    )

    if not path.exists():

        continue

    df = pd.read_csv(
        path
    )

    print(
        "Shape:",
        df.shape
    )

    print(
        "Columns:"
    )

    print(
        df.columns.tolist()
    )

    print(
        "\nFirst 5 rows:"
    )

    print(
        df.head()
        .to_string(
            index=False
        )
    )


# ============================================================
# Inspect coverage of the frozen PV results specifically
# for the new 53-system cohort.
# ============================================================

perf_path = (
    ACH_DIR
    /
    "pv_model_performance.csv"
)


if perf_path.exists():

    perf = pd.read_csv(
        perf_path
    )

    print("\n")
    print("=" * 120)
    print("FROZEN ACHIEVEMENT PERFORMANCE — BASIC QA")
    print("=" * 120)

    if "Country" in perf.columns:

        print(
            "Countries:",
            perf[
                "Country"
            ]
            .nunique()
        )

    if "Outcome" in perf.columns:

        print(
            "\nRows by outcome:"
        )

        print(
            perf[
                "Outcome"
            ]
            .value_counts()
            .sort_index()
            .to_string()
        )

    possible_pv_columns = [
        column
        for column
        in perf.columns
        if (
            "PV"
            in column.upper()
            or
            "RUN"
            in column.upper()
        )
    ]

    print(
        "\nPossible PV/run columns:",
        possible_pv_columns
    )


print("\n")
print("=" * 120)
print("ACHIEVEMENT OUTPUT INSPECTION COMPLETE")
print("=" * 120)



pv_model_performance.csv
Exists: True
Shape: (3690, 10)
Columns:
['PV', 'Country', 'Outcome', 'N', 'R2', 'RMSE', 'MAE', 'ROC_AUC', 'PR_AUC', 'AchievementFamilyShare']

First 5 rows:
 PV Country      Outcome     N       R2     RMSE      MAE  ROC_AUC  PR_AUC  AchievementFamilyShare
  1     ALB BEINGBULLIED  5899 0.115104 0.935813 0.730402      NaN     NaN                0.192087
  1     ARE BEINGBULLIED 16856 0.218047 1.005196 0.806971      NaN     NaN                0.149937
  1     ARG BEINGBULLIED  7770 0.138962 1.006453 0.819332      NaN     NaN                0.168738
  1     AUS BEINGBULLIED 10580 0.214332 0.992622 0.810034      NaN     NaN                0.097442
  1     AUT BEINGBULLIED  5274 0.125612 0.910804 0.728084      NaN     NaN                0.161768


pv_model_rankings.csv
Exists: True
Shape: (62730, 29)
Columns:
['PV', 'Country', 'Outcome', 'Model_type', 'SHAP_space', 'Feature', 'MeanAbsSHAP', 'ImportanceShare', 'MeanSignedSHAP', 'Rank', 'MeanFoldRank', 'MedianFoldRa

In [6]:
# ============================================================
# PHASE 3E — MATCHED CORE17 vs EXPANDED46
# ACHIEVEMENT-CONTROLLED EXPANDED-PREDICTOR VALIDATION
# ============================================================
#
# CORE17 IS NOT REFIT.
#
# We reuse the frozen 10-PV achievement analysis:
#
#   Core14
#   + PViREAD
#   + PViMATH
#   + PViSCIE
#
#
# We fit ONLY:
#
#   Expanded43
#   + PViREAD
#   + PViMATH
#   + PViSCIE
#
# for i = 1,...,10.
#
#
# KEY QUESTION
# ------------
# Does the original predictor backbone remain recognizable
# when BOTH:
#
#   1. the broader contextual predictor space
#   2. achievement
#
# are allowed to compete simultaneously?
#
#
# MATCHING
# --------
# Same:
#   - system
#   - outcome
#   - students
#   - weights
#   - school folds
#   - plausible-value replicate
#
#
# NO STUDENT-LEVEL AVERAGING OF PLAUSIBLE VALUES.
# ============================================================


from pathlib import Path
import pickle
import gc
import time
import re

import numpy as np
import pandas as pd

import xgboost as xgb

from xgboost import (
    XGBRegressor,
    XGBClassifier,
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    roc_auc_score,
    average_precision_score,
)

from scipy.stats import spearmanr


# ============================================================
# 1. Paths
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "expanded_predictor_analysis"
)


ACH_DIR = (
    RESULT_ROOT
    / "achievement_pv"
)


CHECKPOINT_DIR = (
    OUT_DIR
    / "core17_vs_expanded46_checkpoints"
)


FOLD_DIR = (
    OUT_DIR
    / "frozen_primary53_folds"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. Frozen input files
# ============================================================

MODEL_UNIVERSE_PATH = (
    OUT_DIR
    / "frozen_primary53_country_outcome_models.csv"
)


FROZEN_CORE17_PERFORMANCE_PATH = (
    ACH_DIR
    / "pv_model_performance.csv"
)


FROZEN_CORE17_RANKINGS_PATH = (
    ACH_DIR
    / "pv_model_rankings.csv"
)


# ============================================================
# 3. Columns
# ============================================================

SCHOOL_COL = "CNTSCHID"

STUDENT_COL = "CNTSTUID"

WEIGHT_COL = "W_FSTUWT"


# ============================================================
# 4. Predictor universes
# ============================================================

EXPANDED43 = [
    predictor
    for predictor
    in EXPANDED_NONACHIEVEMENT
    if predictor != "PA006Q09TA"
]


assert len(
    EXPANDED43
) == 43


CORE17_CONCEPTUAL = (
    CORE14
    +
    [
        "PV_READ",
        "PV_MATH",
        "PV_SCIE",
    ]
)


EXPANDED46_CONCEPTUAL = (
    EXPANDED43
    +
    [
        "PV_READ",
        "PV_MATH",
        "PV_SCIE",
    ]
)


NEW29 = [
    feature
    for feature
    in EXPANDED43
    if feature not in CORE14
]


assert len(
    CORE17_CONCEPTUAL
) == 17


assert len(
    EXPANDED46_CONCEPTUAL
) == 46


assert len(
    NEW29
) == 29


# ============================================================
# 5. Frozen XGBoost specification
# ============================================================

XGB_COMMON = {

    "n_estimators":
        300,

    "max_depth":
        4,

    "learning_rate":
        0.05,

    "subsample":
        0.8,

    "colsample_bytree":
        0.8,

    "min_child_weight":
        5,

    "reg_lambda":
        4.0,

    "reg_alpha":
        0.6,

    "gamma":
        0.0,

    "random_state":
        42,

    "n_jobs":
        -1,

    "tree_method":
        "hist",
}


# ============================================================
# 6. Load frozen model universe + Core17 outputs
# ============================================================

model_universe = pd.read_csv(
    MODEL_UNIVERSE_PATH
)


core17_perf = pd.read_csv(
    FROZEN_CORE17_PERFORMANCE_PATH
)


core17_rankings = pd.read_csv(
    FROZEN_CORE17_RANKINGS_PATH
)


PRIMARY53 = sorted(
    model_universe[
        "Country"
    ]
    .unique()
)


# Restrict frozen achievement results to Primary53

core17_perf = (
    core17_perf[
        core17_perf[
            "Country"
        ]
        .isin(
            PRIMARY53
        )
    ]
    .merge(
        model_universe[
            [
                "Country",
                "Outcome"
            ]
        ],
        on=[
            "Country",
            "Outcome"
        ],
        how="inner",
        validate="many_to_one"
    )
)


core17_rankings = (
    core17_rankings[
        core17_rankings[
            "Country"
        ]
        .isin(
            PRIMARY53
        )
    ]
    .merge(
        model_universe[
            [
                "Country",
                "Outcome"
            ]
        ],
        on=[
            "Country",
            "Outcome"
        ],
        how="inner",
        validate="many_to_one"
    )
)


# ============================================================
# 7. Canonical conceptual feature names
# ============================================================

def canonical_feature(
    raw_feature,
    conceptual_feature=None
):

    raw = str(
        raw_feature
    )


    if re.fullmatch(
        r"PV\d+READ",
        raw
    ):

        return "PV_READ"


    if re.fullmatch(
        r"PV\d+MATH",
        raw
    ):

        return "PV_MATH"


    if re.fullmatch(
        r"PV\d+SCIE",
        raw
    ):

        return "PV_SCIE"


    if (
        conceptual_feature is not None
        and
        pd.notna(
            conceptual_feature
        )
    ):

        conceptual = str(
            conceptual_feature
        )


        if conceptual.strip() != "":

            # Protect against alternate achievement labels
            if "READ" in conceptual.upper() and raw.startswith("PV"):

                return "PV_READ"


            if "MATH" in conceptual.upper() and raw.startswith("PV"):

                return "PV_MATH"


            if (
                (
                    "SCIE"
                    in conceptual.upper()
                )
                or
                (
                    "SCIENCE"
                    in conceptual.upper()
                )
            ) and raw.startswith("PV"):

                return "PV_SCIE"


            return conceptual


    return raw


core17_rankings[
    "CanonicalFeature"
] = [
    canonical_feature(
        raw_feature=row[
            "Feature"
        ],
        conceptual_feature=row[
            "ConceptualFeature"
        ],
    )
    for _, row
    in core17_rankings.iterrows()
]


# ============================================================
# 8. PRE-FIT ALIGNMENT QA
# ============================================================

expected_performance_rows = (
    len(
        model_universe
    )
    *
    10
)


print("\n")
print("=" * 125)
print("CORE17 FROZEN OUTPUT ALIGNMENT QA")
print("=" * 125)


print(
    "Primary country × outcome models:",
    len(
        model_universe
    )
)


print(
    "Expected PV performance rows:",
    expected_performance_rows
)


print(
    "Observed frozen Core17 performance rows:",
    len(
        core17_perf
    )
)


if len(
    core17_perf
) != expected_performance_rows:

    raise RuntimeError(
        "Frozen Core17 performance coverage does not "
        "match expected Primary53 × 10-PV universe."
    )


# Every Country × Outcome should have PV 1...10

pv_count_qa = (
    core17_perf
    .groupby(
        [
            "Country",
            "Outcome"
        ]
    )[
        "PV"
    ]
    .nunique()
)


print(
    "Country × outcome profiles with exactly 10 PVs:",
    int(
        (
            pv_count_qa
            ==
            10
        )
        .sum()
    ),
    "/",
    len(
        pv_count_qa
    )
)


if not (
    pv_count_qa
    ==
    10
).all():

    raise RuntimeError(
        "Some frozen Core17 profiles do not contain 10 PVs."
    )


# ============================================================
# 9. Ranking QA
# ============================================================

concept_count_qa = (
    core17_rankings
    .groupby(
        [
            "PV",
            "Country",
            "Outcome"
        ]
    )[
        "CanonicalFeature"
    ]
    .nunique()
)


print(
    "PV profiles with exactly 17 conceptual features:",
    int(
        (
            concept_count_qa
            ==
            17
        )
        .sum()
    ),
    "/",
    len(
        concept_count_qa
    )
)


if not (
    concept_count_qa
    ==
    17
).all():

    bad = (
        concept_count_qa[
            concept_count_qa
            !=
            17
        ]
    )


    print(
        bad.head(
            20
        )
    )


    raise RuntimeError(
        "Frozen Core17 ranking profiles do not map cleanly "
        "to 17 conceptual predictors."
    )


frozen_concepts = sorted(
    core17_rankings[
        "CanonicalFeature"
    ]
    .unique()
)


print(
    "\nFrozen Core17 conceptual features:"
)


print(
    frozen_concepts
)


if set(
    frozen_concepts
) != set(
    CORE17_CONCEPTUAL
):

    raise RuntimeError(
        "Frozen Core17 conceptual feature set differs from "
        "expected Core17."
    )


# ============================================================
# 10. Check sample N against Phase 3C
# ============================================================

sample_audit = pd.read_csv(
    OUT_DIR
    / "expanded_primary53_outcome_sample_audit.csv"
)


sample_n = (
    sample_audit[
        sample_audit[
            "BasicEligible"
        ]
    ][
        [
            "Country",
            "Outcome",
            "N_Students"
        ]
    ]
    .rename(
        columns={
            "N_Students":
                "ExpectedN"
        }
    )
)


n_alignment = (
    core17_perf
    .merge(
        sample_n,
        on=[
            "Country",
            "Outcome"
        ],
        how="left",
        validate="many_to_one"
    )
)


n_alignment[
    "N_Match"
] = (
    n_alignment[
        "N"
    ]
    ==
    n_alignment[
        "ExpectedN"
    ]
)


print(
    "\nFrozen Core17 sample-N matches:",
    int(
        n_alignment[
            "N_Match"
        ]
        .sum()
    ),
    "/",
    len(
        n_alignment
    )
)


if not n_alignment[
    "N_Match"
].all():

    print(
        n_alignment[
            ~n_alignment[
                "N_Match"
            ]
        ][
            [
                "PV",
                "Country",
                "Outcome",
                "N",
                "ExpectedN"
            ]
        ]
        .head(
            30
        )
        .to_string(
            index=False
        )
    )


    raise RuntimeError(
        "Frozen Core17 student samples differ from Phase 3C."
    )


print(
    "\nAlignment QA passed."
)


# ============================================================
# 11. Helper: numeric predictor frame
# ============================================================

def numeric_frame(
    df,
    features
):

    output = pd.DataFrame(
        index=df.index
    )


    for feature in features:

        output[
            feature
        ] = pd.to_numeric(
            df[
                feature
            ],
            errors="coerce"
        )


    return output


# ============================================================
# 12. Model constructor
# ============================================================

def make_model(
    outcome
):

    if outcome == "BEINGBULLIED":

        return XGBRegressor(
            objective=
                "reg:squarederror",
            **XGB_COMMON
        )


    return XGBClassifier(
        objective=
            "binary:logistic",
        eval_metric=
            "logloss",
        **XGB_COMMON
    )


# ============================================================
# 13. Prediction
# ============================================================

def predict_model(
    model,
    X,
    outcome
):

    if outcome == "BEINGBULLIED":

        return model.predict(
            X
        )


    return model.predict_proba(
        X
    )[
        :,
        1
    ]


# ============================================================
# 14. Weighted performance
# ============================================================

def calculate_performance(
    y,
    pred,
    weights,
    outcome
):

    y = np.asarray(
        y,
        dtype=float
    )


    pred = np.asarray(
        pred,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    if outcome == "BEINGBULLIED":

        residual2 = (
            y
            -
            pred
        ) ** 2


        rmse = float(
            np.sqrt(
                np.average(
                    residual2,
                    weights=weights
                )
            )
        )


        return {

            "R2":
                float(
                    r2_score(
                        y,
                        pred,
                        sample_weight=weights
                    )
                ),

            "RMSE":
                rmse,

            "MAE":
                float(
                    mean_absolute_error(
                        y,
                        pred,
                        sample_weight=weights
                    )
                ),

            "ROC_AUC":
                np.nan,

            "PR_AUC":
                np.nan,
        }


    return {

        "R2":
            np.nan,

        "RMSE":
            np.nan,

        "MAE":
            np.nan,

        "ROC_AUC":
            float(
                roc_auc_score(
                    y,
                    pred,
                    sample_weight=weights
                )
            ),

        "PR_AUC":
            float(
                average_precision_score(
                    y,
                    pred,
                    sample_weight=weights
                )
            ),
    }


# ============================================================
# 15. Native exact TreeSHAP
# ============================================================

def native_shap_values(
    model,
    X,
    features
):

    dmatrix = xgb.DMatrix(
        X,
        feature_names=features,
        missing=np.nan
    )


    contributions = (
        model
        .get_booster()
        .predict(
            dmatrix,
            pred_contribs=True,
            approx_contribs=False
        )
    )


    shap_values = (
        contributions[
            :,
            :-1
        ]
    )


    if (
        shap_values.shape[
            1
        ]
        !=
        len(
            features
        )
    ):

        raise RuntimeError(
            "Unexpected TreeSHAP matrix width."
        )


    return shap_values


# ============================================================
# 16. Fit one Expanded46 PV model
# ============================================================

def fit_expanded46_oof(
    X,
    y,
    weights,
    fold_ids,
    outcome,
    features
):

    n = len(
        y
    )


    oof_predictions = np.full(
        n,
        np.nan,
        dtype=float
    )


    weighted_abs_shap_sum = np.zeros(
        len(
            features
        ),
        dtype=float
    )


    shap_weight_sum = 0.0


    for fold in range(
        5
    ):

        train_mask = (
            fold_ids
            !=
            fold
        )


        test_mask = (
            fold_ids
            ==
            fold
        )


        X_train = (
            X.loc[
                train_mask,
                features
            ]
        )


        X_test = (
            X.loc[
                test_mask,
                features
            ]
        )


        y_train = y[
            train_mask
        ]


        y_test = y[
            test_mask
        ]


        w_train = weights[
            train_mask
        ]


        w_test = weights[
            test_mask
        ]


        if outcome != "BEINGBULLIED":

            if len(
                np.unique(
                    y_train
                )
            ) != 2:

                raise RuntimeError(
                    "Training fold contains one class."
                )


            if len(
                np.unique(
                    y_test
                )
            ) != 2:

                raise RuntimeError(
                    "Test fold contains one class."
                )


        w_train_norm = (
            w_train
            /
            np.mean(
                w_train
            )
        )


        model = make_model(
            outcome
        )


        model.fit(
            X_train,
            y_train,
            sample_weight=
                w_train_norm
        )


        pred = predict_model(
            model,
            X_test,
            outcome
        )


        oof_predictions[
            test_mask
        ] = pred


        shap_values = native_shap_values(
            model,
            X_test,
            features
        )


        weighted_abs_shap_sum += (
            np.sum(
                np.abs(
                    shap_values
                )
                *
                w_test[
                    :,
                    None
                ],
                axis=0
            )
        )


        shap_weight_sum += float(
            np.sum(
                w_test
            )
        )


        del (
            model,
            X_train,
            X_test,
            shap_values
        )


        gc.collect()


    if np.isnan(
        oof_predictions
    ).any():

        raise RuntimeError(
            "Missing OOF predictions."
        )


    performance = calculate_performance(
        y,
        oof_predictions,
        weights,
        outcome
    )


    mean_abs_shap = (
        weighted_abs_shap_sum
        /
        shap_weight_sum
    )


    total_shap = float(
        mean_abs_shap.sum()
    )


    if (
        not np.isfinite(
            total_shap
        )
        or
        total_shap
        <=
        0
    ):

        raise RuntimeError(
            "Invalid total SHAP attribution."
        )


    share = (
        mean_abs_shap
        /
        total_shap
    )


    attribution = pd.DataFrame(
        {
            "Feature":
                features,

            "MeanAbsSHAP":
                mean_abs_shap,

            "ImportanceShare":
                share,
        }
    )


    attribution = (
        attribution
        .sort_values(
            "ImportanceShare",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    attribution[
        "Rank"
    ] = (
        np.arange(
            1,
            len(
                attribution
            )
            +
            1
        )
    )


    return (
        performance,
        attribution
    )


# ============================================================
# 17. Similarity helpers
# ============================================================

def cosine_similarity(
    a,
    b
):

    a = np.asarray(
        a,
        dtype=float
    )


    b = np.asarray(
        b,
        dtype=float
    )


    denominator = (
        np.linalg.norm(
            a
        )
        *
        np.linalg.norm(
            b
        )
    )


    if denominator == 0:

        return np.nan


    return float(
        np.dot(
            a,
            b
        )
        /
        denominator
    )


# ============================================================
# 18. Compare frozen Core17 vs Expanded46 architecture
# ============================================================

def compare_core17_expanded46(
    core17_attr,
    expanded46_attr
):

    core = (
        core17_attr
        .set_index(
            "CanonicalFeature"
        )
        .loc[
            CORE17_CONCEPTUAL
        ]
        .copy()
    )


    expanded = (
        expanded46_attr
        .set_index(
            "CanonicalFeature"
        )
        .copy()
    )


    expanded_core = (
        expanded
        .loc[
            CORE17_CONCEPTUAL
        ]
        .copy()
    )


    core_mass = float(
        expanded_core[
            "ImportanceShare"
        ]
        .sum()
    )


    expanded_core_renorm = (
        expanded_core[
            "ImportanceShare"
        ]
        /
        core_mass
    )


    original_core_share = (
        core[
            "ImportanceShare"
        ]
    )


    cosine = cosine_similarity(
        original_core_share,
        expanded_core_renorm
    )


    rank_core = (
        original_core_share
        .rank(
            ascending=False,
            method="average"
        )
    )


    rank_expanded = (
        expanded_core_renorm
        .rank(
            ascending=False,
            method="average"
        )
    )


    rho = spearmanr(
        rank_core,
        rank_expanded
    ).statistic


    global_ranks = (
        expanded[
            "Rank"
        ]
    )


    n_core_top5 = int(
        (
            global_ranks
            .loc[
                CORE17_CONCEPTUAL
            ]
            <=
            5
        )
        .sum()
    )


    n_core_top10 = int(
        (
            global_ranks
            .loc[
                CORE17_CONCEPTUAL
            ]
            <=
            10
        )
        .sum()
    )


    belong_rank = int(
        expanded.loc[
            "BELONG",
            "Rank"
        ]
    )


    belong_share = float(
        expanded.loc[
            "BELONG",
            "ImportanceShare"
        ]
    )


    achievement_share = float(
        expanded.loc[
            [
                "PV_READ",
                "PV_MATH",
                "PV_SCIE"
            ],
            "ImportanceShare"
        ]
        .sum()
    )


    achievement_ranks = (
        expanded.loc[
            [
                "PV_READ",
                "PV_MATH",
                "PV_SCIE"
            ],
            "Rank"
        ]
        .astype(
            int
        )
    )


    n_new_top10 = int(
        (
            global_ranks
            .loc[
                NEW29
            ]
            <=
            10
        )
        .sum()
    )


    new_top10 = (
        global_ranks
        .loc[
            NEW29
        ]
    )


    new_top10 = (
        new_top10[
            new_top10
            <=
            10
        ]
        .sort_values()
    )


    return {

        "Core17AttributionMass":
            core_mass,

        "Core17RenormCosine":
            cosine,

        "Core17SharedRankSpearman":
            float(
                rho
            ),

        "N_Core17_in_Top5":
            n_core_top5,

        "N_Core17_in_Top10":
            n_core_top10,

        "BELONG_ExpandedRank":
            belong_rank,

        "BELONG_ExpandedShare":
            belong_share,

        "ExpandedAchievementFamilyShare":
            achievement_share,

        "PV_READ_Rank":
            int(
                achievement_ranks[
                    "PV_READ"
                ]
            ),

        "PV_MATH_Rank":
            int(
                achievement_ranks[
                    "PV_MATH"
                ]
            ),

        "PV_SCIE_Rank":
            int(
                achievement_ranks[
                    "PV_SCIE"
                ]
            ),

        "N_New29_in_Top10":
            n_new_top10,

        "NewTop10Features":
            "|".join(
                new_top10.index.tolist()
            ),
    }


# ============================================================
# 19. MAIN LOOP
#
# One checkpoint per Country × Outcome.
# Each checkpoint contains all 10 PVs.
# ============================================================

start_time = time.time()


total_pairs = len(
    model_universe
)


for pair_number, pair in (
    model_universe
    .reset_index(
        drop=True
    )
    .iterrows()
):

    country = pair[
        "Country"
    ]


    outcome = pair[
        "Outcome"
    ]


    checkpoint_path = (
        CHECKPOINT_DIR
        /
        f"{country}__{outcome}.pkl"
    )


    if checkpoint_path.exists():

        print(
            f"[{pair_number + 1:03d}/{total_pairs}] "
            f"{country} {outcome} — already complete"
        )


        continue


    print(
        "\n"
        +
        "=" * 100
    )


    print(
        f"[{pair_number + 1:03d}/{total_pairs}] "
        f"{country} | {outcome}"
    )


    pair_start = time.time()


    # --------------------------------------------------------
    # Load processed system file
    # --------------------------------------------------------

    path = (
        PROCESSED_DIR
        /
        f"{country}_analysis.pkl"
    )


    df = pd.read_pickle(
        path
    )


    # --------------------------------------------------------
    # Exact outcome-valid sample
    # --------------------------------------------------------

    y_full = pd.to_numeric(
        df[
            outcome
        ],
        errors="coerce"
    )


    weight_full = pd.to_numeric(
        df[
            WEIGHT_COL
        ],
        errors="coerce"
    )


    valid_mask = (

        np.isfinite(
            y_full
        )

        &

        np.isfinite(
            weight_full
        )

        &

        (
            weight_full
            >
            0
        )

        &

        df[
            SCHOOL_COL
        ]
        .notna()
    )


    selected = (
        df.loc[
            valid_mask
        ]
        .copy()
    )


    y = (
        pd.to_numeric(
            selected[
                outcome
            ],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    weights = (
        pd.to_numeric(
            selected[
                WEIGHT_COL
            ],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    student_ids = (
        selected[
            STUDENT_COL
        ]
        .astype(
            str
        )
        .to_numpy()
    )


    # --------------------------------------------------------
    # Load EXACT frozen fold assignment from Phase 3D
    # --------------------------------------------------------

    fold_path = (
        FOLD_DIR
        /
        f"{country}__{outcome}__folds.pkl"
    )


    if not fold_path.exists():

        raise FileNotFoundError(
            f"Frozen fold assignment missing: {fold_path}"
        )


    fold_table = pd.read_pickle(
        fold_path
    )


    fold_map = (
        fold_table
        .set_index(
            "CNTSTUID"
        )[
            "Fold"
        ]
    )


    if not set(
        student_ids
    ).issubset(
        set(
            fold_map.index
        )
    ):

        raise RuntimeError(
            f"Student IDs do not match frozen folds "
            f"for {country} {outcome}."
        )


    fold_ids = (
        pd.Series(
            student_ids
        )
        .map(
            fold_map
        )
        .to_numpy(
            dtype=int
        )
    )


    if (
        fold_ids
        <
        0
    ).any():

        raise RuntimeError(
            "Invalid frozen fold ID."
        )


    # --------------------------------------------------------
    # Non-achievement part only once
    # --------------------------------------------------------

    X_base43 = numeric_frame(
        selected,
        EXPANDED43
    )


    pv_results = []


    for pv in range(
        1,
        11
    ):

        print(
            f"   PV {pv}/10",
            end=""
        )


        # ====================================================
        # Add matched achievement trio
        # ====================================================

        read_col = (
            f"PV{pv}READ"
        )


        math_col = (
            f"PV{pv}MATH"
        )


        scie_col = (
            f"PV{pv}SCIE"
        )


        model_features = (
            EXPANDED43
            +
            [
                read_col,
                math_col,
                scie_col,
            ]
        )


        assert len(
            model_features
        ) == 46


        X = X_base43.copy()


        X[
            read_col
        ] = pd.to_numeric(
            selected[
                read_col
            ],
            errors="coerce"
        )


        X[
            math_col
        ] = pd.to_numeric(
            selected[
                math_col
            ],
            errors="coerce"
        )


        X[
            scie_col
        ] = pd.to_numeric(
            selected[
                scie_col
            ],
            errors="coerce"
        )


        # ====================================================
        # Fit Expanded46
        # ====================================================

        expanded_perf, expanded_attr = (
            fit_expanded46_oof(
                X=X,
                y=y,
                weights=weights,
                fold_ids=fold_ids,
                outcome=outcome,
                features=model_features,
            )
        )


        # ====================================================
        # Canonicalize Expanded46 features
        # ====================================================

        expanded_attr[
            "CanonicalFeature"
        ] = [
            canonical_feature(
                raw_feature=feature
            )
            for feature
            in expanded_attr[
                "Feature"
            ]
        ]


        if (
            expanded_attr[
                "CanonicalFeature"
            ]
            .nunique()
            !=
            46
        ):

            raise RuntimeError(
                "Expanded46 did not map to 46 unique "
                "conceptual predictors."
            )


        # ====================================================
        # Retrieve EXACT frozen Core17 PV result
        # ====================================================

        frozen_perf = (
            core17_perf[
                (
                    core17_perf[
                        "PV"
                    ]
                    ==
                    pv
                )
                &
                (
                    core17_perf[
                        "Country"
                    ]
                    ==
                    country
                )
                &
                (
                    core17_perf[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
            ]
        )


        if len(
            frozen_perf
        ) != 1:

            raise RuntimeError(
                "Expected exactly one frozen Core17 "
                "performance row."
            )


        frozen_perf = (
            frozen_perf
            .iloc[
                0
            ]
        )


        frozen_attr = (
            core17_rankings[
                (
                    core17_rankings[
                        "PV"
                    ]
                    ==
                    pv
                )
                &
                (
                    core17_rankings[
                        "Country"
                    ]
                    ==
                    country
                )
                &
                (
                    core17_rankings[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
            ]
            .copy()
        )


        if len(
            frozen_attr
        ) != 17:

            raise RuntimeError(
                "Expected 17 frozen Core17 ranking rows."
            )


        # ====================================================
        # Architecture preservation
        # ====================================================

        architecture = (
            compare_core17_expanded46(
                core17_attr=
                    frozen_attr,

                expanded46_attr=
                    expanded_attr,
            )
        )


        # ====================================================
        # Performance comparison
        # ====================================================

        if outcome == "BEINGBULLIED":

            primary_metric = (
                "R2"
            )


            core_metric = float(
                frozen_perf[
                    "R2"
                ]
            )


            expanded_metric = float(
                expanded_perf[
                    "R2"
                ]
            )


        else:

            primary_metric = (
                "ROC_AUC"
            )


            core_metric = float(
                frozen_perf[
                    "ROC_AUC"
                ]
            )


            expanded_metric = float(
                expanded_perf[
                    "ROC_AUC"
                ]
            )


        # ====================================================
        # Store attribution with IDs
        # ====================================================

        expanded_attr = (
            expanded_attr.copy()
        )


        expanded_attr.insert(
            0,
            "PV",
            pv
        )


        expanded_attr.insert(
            0,
            "Outcome",
            outcome
        )


        expanded_attr.insert(
            0,
            "Country",
            country
        )


        pv_results.append(
            {

                "PV":
                    pv,

                "Country":
                    country,

                "Outcome":
                    outcome,

                "N":
                    len(
                        y
                    ),

                "CorePrimaryMetric":
                    core_metric,

                "ExpandedPrimaryMetric":
                    expanded_metric,

                "DeltaPrimaryMetric":
                    (
                        expanded_metric
                        -
                        core_metric
                    ),

                "PrimaryMetric":
                    primary_metric,

                "CoreR2":
                    frozen_perf[
                        "R2"
                    ],

                "CoreROC_AUC":
                    frozen_perf[
                        "ROC_AUC"
                    ],

                "CorePR_AUC":
                    frozen_perf[
                        "PR_AUC"
                    ],

                "CoreAchievementFamilyShare":
                    frozen_perf[
                        "AchievementFamilyShare"
                    ],

                "ExpandedR2":
                    expanded_perf[
                        "R2"
                    ],

                "ExpandedRMSE":
                    expanded_perf[
                        "RMSE"
                    ],

                "ExpandedMAE":
                    expanded_perf[
                        "MAE"
                    ],

                "ExpandedROC_AUC":
                    expanded_perf[
                        "ROC_AUC"
                    ],

                "ExpandedPR_AUC":
                    expanded_perf[
                        "PR_AUC"
                    ],

                **architecture,

                "ExpandedAttribution":
                    expanded_attr,
            }
        )


        print(
            " | Δ=",
            round(
                expanded_metric
                -
                core_metric,
                4
            ),
            "| BELONG rank=",
            architecture[
                "BELONG_ExpandedRank"
            ],
            "| Core17 mass=",
            round(
                architecture[
                    "Core17AttributionMass"
                ],
                3
            )
        )


        del (
            X,
            expanded_attr
        )


        gc.collect()


    # --------------------------------------------------------
    # Save one checkpoint after all 10 PVs
    # --------------------------------------------------------

    with open(
        checkpoint_path,
        "wb"
    ) as file:

        pickle.dump(
            pv_results,
            file
        )


    elapsed = (
        time.time()
        -
        pair_start
    )


    print(
        f"   Pair complete in {elapsed / 60:.2f} min"
    )


    del (
        df,
        selected,
        X_base43,
        pv_results
    )


    gc.collect()


# ============================================================
# 20. Reassemble checkpoints
# ============================================================

comparison_rows = []

expanded_attribution_rows = []


for _, pair in (
    model_universe.iterrows()
):

    country = pair[
        "Country"
    ]


    outcome = pair[
        "Outcome"
    ]


    checkpoint_path = (
        CHECKPOINT_DIR
        /
        f"{country}__{outcome}.pkl"
    )


    if not checkpoint_path.exists():

        raise RuntimeError(
            f"Missing checkpoint: {country} {outcome}"
        )


    with open(
        checkpoint_path,
        "rb"
    ) as file:

        results = pickle.load(
            file
        )


    if len(
        results
    ) != 10:

        raise RuntimeError(
            "Checkpoint does not contain all 10 PVs."
        )


    for result in results:

        attr = result[
            "ExpandedAttribution"
        ]


        expanded_attribution_rows.append(
            attr
        )


        comparison_rows.append(
            {
                key:
                    value
                for key, value
                in result.items()
                if key
                !=
                "ExpandedAttribution"
            }
        )


comparison_df = pd.DataFrame(
    comparison_rows
)


expanded_attr_df = pd.concat(
    expanded_attribution_rows,
    ignore_index=True
)


# ============================================================
# 21. QA
# ============================================================

expected_rows = (
    len(
        model_universe
    )
    *
    10
)


if len(
    comparison_df
) != expected_rows:

    raise RuntimeError(
        "Unexpected comparison row count."
    )


share_qa = (
    expanded_attr_df
    .groupby(
        [
            "PV",
            "Country",
            "Outcome"
        ]
    )[
        "ImportanceShare"
    ]
    .sum()
)


max_share_error = float(
    np.abs(
        share_qa
        -
        1
    )
    .max()
)


print(
    "\nMaximum Expanded46 SHAP share error:",
    max_share_error
)


# ============================================================
# 22. Save PV-level outputs
# ============================================================

comparison_df.to_csv(
    OUT_DIR
    / "core17_vs_expanded46_pv_comparison.csv",
    index=False
)


expanded_attr_df.to_csv(
    OUT_DIR
    / "expanded46_pv_oof_shap_rankings.csv",
    index=False
)


# ============================================================
# 23. First aggregate:
# average each system across its 10 plausible values
# ============================================================

system_comparison = (
    comparison_df
    .groupby(
        [
            "Country",
            "Outcome",
            "PrimaryMetric"
        ],
        as_index=False
    )
    .agg(

        CorePrimaryMetric=(
            "CorePrimaryMetric",
            "mean"
        ),

        ExpandedPrimaryMetric=(
            "ExpandedPrimaryMetric",
            "mean"
        ),

        DeltaPrimaryMetric=(
            "DeltaPrimaryMetric",
            "mean"
        ),

        Core17AttributionMass=(
            "Core17AttributionMass",
            "mean"
        ),

        Core17RenormCosine=(
            "Core17RenormCosine",
            "mean"
        ),

        Core17SharedRankSpearman=(
            "Core17SharedRankSpearman",
            "mean"
        ),

        N_Core17_in_Top5=(
            "N_Core17_in_Top5",
            "mean"
        ),

        N_Core17_in_Top10=(
            "N_Core17_in_Top10",
            "mean"
        ),

        BELONG_ExpandedRank=(
            "BELONG_ExpandedRank",
            "mean"
        ),

        BELONG_ExpandedShare=(
            "BELONG_ExpandedShare",
            "mean"
        ),

        CoreAchievementFamilyShare=(
            "CoreAchievementFamilyShare",
            "mean"
        ),

        ExpandedAchievementFamilyShare=(
            "ExpandedAchievementFamilyShare",
            "mean"
        ),

        PV_READ_Rank=(
            "PV_READ_Rank",
            "mean"
        ),

        PV_MATH_Rank=(
            "PV_MATH_Rank",
            "mean"
        ),

        PV_SCIE_Rank=(
            "PV_SCIE_Rank",
            "mean"
        ),

        N_New29_in_Top10=(
            "N_New29_in_Top10",
            "mean"
        ),
    )
)


system_comparison.to_csv(
    OUT_DIR
    / "core17_vs_expanded46_system_comparison.csv",
    index=False
)


# ============================================================
# 24. Performance summary across systems
# ============================================================

performance_summary = (
    system_comparison
    .groupby(
        "Outcome",
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        Mean_Core=(
            "CorePrimaryMetric",
            "mean"
        ),

        Mean_Expanded=(
            "ExpandedPrimaryMetric",
            "mean"
        ),

        Mean_Delta=(
            "DeltaPrimaryMetric",
            "mean"
        ),

        Median_Delta=(
            "DeltaPrimaryMetric",
            "median"
        ),

        Min_Delta=(
            "DeltaPrimaryMetric",
            "min"
        ),

        Max_Delta=(
            "DeltaPrimaryMetric",
            "max"
        ),

        N_ExpandedBetter=(
            "DeltaPrimaryMetric",
            lambda x:
                int(
                    (
                        x
                        >
                        0
                    )
                    .sum()
                )
        ),

        Pct_ExpandedBetter=(
            "DeltaPrimaryMetric",
            lambda x:
                float(
                    (
                        x
                        >
                        0
                    )
                    .mean()
                    *
                    100
                )
        ),
    )
)


performance_summary.to_csv(
    OUT_DIR
    / "core17_vs_expanded46_performance_summary.csv",
    index=False
)


# ============================================================
# 25. Architecture-preservation summary
# ============================================================

preservation_summary = (
    system_comparison
    .groupby(
        "Outcome",
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        Mean_Core17_AttributionMass=(
            "Core17AttributionMass",
            "mean"
        ),

        Median_Core17_AttributionMass=(
            "Core17AttributionMass",
            "median"
        ),

        Mean_Core17_RenormCosine=(
            "Core17RenormCosine",
            "mean"
        ),

        Mean_Core17_RankSpearman=(
            "Core17SharedRankSpearman",
            "mean"
        ),

        Mean_Core_in_Top5=(
            "N_Core17_in_Top5",
            "mean"
        ),

        Mean_Core_in_Top10=(
            "N_Core17_in_Top10",
            "mean"
        ),

        BELONG_MeanRank=(
            "BELONG_ExpandedRank",
            "mean"
        ),

        BELONG_MedianRank=(
            "BELONG_ExpandedRank",
            "median"
        ),

        CoreAchievementShare=(
            "CoreAchievementFamilyShare",
            "mean"
        ),

        ExpandedAchievementShare=(
            "ExpandedAchievementFamilyShare",
            "mean"
        ),

        READ_MeanRank=(
            "PV_READ_Rank",
            "mean"
        ),

        MATH_MeanRank=(
            "PV_MATH_Rank",
            "mean"
        ),

        SCIE_MeanRank=(
            "PV_SCIE_Rank",
            "mean"
        ),

        Mean_New29_in_Top10=(
            "N_New29_in_Top10",
            "mean"
        ),
    )
)


preservation_summary.to_csv(
    OUT_DIR
    / "core17_preservation_summary_expanded46.csv",
    index=False
)


# ============================================================
# 26. Average Expanded46 attribution over PVs within system
# ============================================================

expanded_conceptual = (
    expanded_attr_df
    .groupby(
        [
            "Country",
            "Outcome",
            "CanonicalFeature"
        ],
        as_index=False
    )
    .agg(

        MeanImportanceShare=(
            "ImportanceShare",
            "mean"
        ),

        MeanRankAcrossPVs=(
            "Rank",
            "mean"
        ),

        MedianRankAcrossPVs=(
            "Rank",
            "median"
        ),

        BestRankAcrossPVs=(
            "Rank",
            "min"
        ),

        WorstRankAcrossPVs=(
            "Rank",
            "max"
        ),
    )
)


expanded_conceptual[
    "ConceptualRank"
] = (
    expanded_conceptual
    .groupby(
        [
            "Country",
            "Outcome"
        ]
    )[
        "MeanImportanceShare"
    ]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(
        int
    )
)


expanded_conceptual.to_csv(
    OUT_DIR
    / "expanded46_system_conceptual_predictor_summary.csv",
    index=False
)


# ============================================================
# 27. Global Expanded46 attribution
# Equal system weight
# ============================================================

global_expanded46 = (
    expanded_conceptual
    .groupby(
        [
            "Outcome",
            "CanonicalFeature"
        ],
        as_index=False
    )
    .agg(

        MeanShare=(
            "MeanImportanceShare",
            "mean"
        ),

        MeanRank=(
            "ConceptualRank",
            "mean"
        ),

        MedianRank=(
            "ConceptualRank",
            "median"
        ),

        Rank1Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        ==
                        1
                    )
                    .sum()
                )
        ),

        Top5Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        <=
                        5
                    )
                    .sum()
                )
        ),

        Top10Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        <=
                        10
                    )
                    .sum()
                )
        ),
    )
)


global_expanded46[
    "OriginalCore17"
] = (
    global_expanded46[
        "CanonicalFeature"
    ]
    .isin(
        CORE17_CONCEPTUAL
    )
)


global_expanded46[
    "OriginalCore14"
] = (
    global_expanded46[
        "CanonicalFeature"
    ]
    .isin(
        CORE14
    )
)


global_expanded46 = (
    global_expanded46
    .sort_values(
        [
            "Outcome",
            "MeanShare"
        ],
        ascending=[
            True,
            False
        ]
    )
)


global_expanded46[
    "GlobalRank"
] = (
    global_expanded46
    .groupby(
        "Outcome"
    )
    .cumcount()
    +
    1
)


global_expanded46.to_csv(
    OUT_DIR
    / "expanded46_global_attribution_summary.csv",
    index=False
)


# ============================================================
# 28. New29 entrants after controlling for achievement
# ============================================================

new29_summary = (
    expanded_conceptual[
        expanded_conceptual[
            "CanonicalFeature"
        ]
        .isin(
            NEW29
        )
    ]
    .groupby(
        [
            "Outcome",
            "CanonicalFeature"
        ],
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        MeanShare=(
            "MeanImportanceShare",
            "mean"
        ),

        MeanRank=(
            "ConceptualRank",
            "mean"
        ),

        Rank1Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        ==
                        1
                    )
                    .sum()
                )
        ),

        Top5Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        <=
                        5
                    )
                    .sum()
                )
        ),

        Top10Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        <=
                        10
                    )
                    .sum()
                )
        ),
    )
)


new29_summary = (
    new29_summary
    .sort_values(
        [
            "Outcome",
            "Top10Count",
            "MeanShare"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)


new29_summary.to_csv(
    OUT_DIR
    / "expanded46_new29_entrants.csv",
    index=False
)


# ============================================================
# 29. BELONG survival
# ============================================================

belong_summary = (
    global_expanded46[
        global_expanded46[
            "CanonicalFeature"
        ]
        ==
        "BELONG"
    ]
    .copy()
)


# ============================================================
# 30. Achievement feature summary
# ============================================================

achievement_summary = (
    global_expanded46[
        global_expanded46[
            "CanonicalFeature"
        ]
        .isin(
            [
                "PV_READ",
                "PV_MATH",
                "PV_SCIE"
            ]
        )
    ]
    .copy()
)


# ============================================================
# 31. Print performance
# ============================================================

print("\n")
print("=" * 125)
print("CORE17 vs EXPANDED46 — PERFORMANCE")
print("=" * 125)


print(
    performance_summary
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 32. Print preservation
# ============================================================

print("\n")
print("=" * 125)
print("CORE17 ARCHITECTURE PRESERVATION INSIDE EXPANDED46")
print("=" * 125)


print(
    preservation_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 33. Global top features
# ============================================================

print("\n")
print("=" * 125)
print("TOP EXPANDED46 FEATURES BY OUTCOME")
print("=" * 125)


for outcome in sorted(
    global_expanded46[
        "Outcome"
    ]
    .unique()
):

    print(
        "\n",
        outcome
    )


    print(
        global_expanded46[
            global_expanded46[
                "Outcome"
            ]
            ==
            outcome
        ][
            [
                "GlobalRank",
                "CanonicalFeature",
                "OriginalCore14",
                "OriginalCore17",
                "MeanShare",
                "MeanRank",
                "Rank1Count",
                "Top5Count",
                "Top10Count",
            ]
        ]
        .head(
            15
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 34. New entrant summary
# ============================================================

print("\n")
print("=" * 125)
print("STRONGEST NEW29 ENTRANTS AFTER ACHIEVEMENT CONTROL")
print("=" * 125)


for outcome in sorted(
    new29_summary[
        "Outcome"
    ]
    .unique()
):

    print(
        "\n",
        outcome
    )


    print(
        new29_summary[
            new29_summary[
                "Outcome"
            ]
            ==
            outcome
        ]
        .head(
            12
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 35. BELONG
# ============================================================

print("\n")
print("=" * 125)
print("BELONG SURVIVAL IN EXPANDED46")
print("=" * 125)


print(
    belong_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 36. Achievement predictors
# ============================================================

print("\n")
print("=" * 125)
print("ACHIEVEMENT PREDICTORS INSIDE EXPANDED46")
print("=" * 125)


print(
    achievement_summary
    .sort_values(
        [
            "Outcome",
            "GlobalRank"
        ]
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 37. Final status
# ============================================================

runtime_minutes = (
    time.time()
    -
    start_time
) / 60


print("\n")
print("=" * 125)
print("PHASE 3E CORE17 vs EXPANDED46 COMPLETE")
print("=" * 125)


print(
    "PV-level comparisons:",
    len(
        comparison_df
    )
)


print(
    "System × outcome comparisons:",
    len(
        system_comparison
    )
)


print(
    "Runtime this invocation:",
    round(
        runtime_minutes,
        2
    ),
    "minutes"
)


print(
    "\nFrozen Core17 models were reused, not refit."
)


print(
    "Expanded46 was fitted separately for all 10 matched "
    "plausible-value replicates."
)



CORE17 FROZEN OUTPUT ALIGNMENT QA
Primary country × outcome models: 264
Expected PV performance rows: 2640
Observed frozen Core17 performance rows: 2640
Country × outcome profiles with exactly 10 PVs: 264 / 264
PV profiles with exactly 17 conceptual features: 2640 / 2640

Frozen Core17 conceptual features:
['ADAPTIVITY', 'AGE', 'ATTLNACT', 'BELONG', 'DISCLIMA', 'EDUSHORT', 'ESCS', 'JOYREAD', 'MASTGOAL', 'METASPAM', 'METASUM', 'PV_MATH', 'PV_READ', 'PV_SCIE', 'ST004D01T', 'STIMREAD', 'UNDREM']

Frozen Core17 sample-N matches: 2640 / 2640

Alignment QA passed.

[001/264] ARE | BEINGBULLIED
   PV 1/10 | Δ= 0.06 | BELONG rank= 1 | Core17 mass= 0.555
   PV 2/10 | Δ= 0.0544 | BELONG rank= 1 | Core17 mass= 0.559
   PV 3/10 | Δ= 0.0541 | BELONG rank= 1 | Core17 mass= 0.558
   PV 4/10 | Δ= 0.055 | BELONG rank= 1 | Core17 mass= 0.558
   PV 5/10 | Δ= 0.0528 | BELONG rank= 1 | Core17 mass= 0.555
   PV 6/10 | Δ= 0.0584 | BELONG rank= 1 | Core17 mass= 0.553
   PV 7/10 | Δ= 0.0579 | BELONG rank= 1 

In [7]:
# ============================================================
# PHASE 3F — LITERAL FULL47 PARENT-MODULE SENSITIVITY
#
# Expanded46 vs Full47
#
# ONLY in systems where PA006Q09TA passed the frozen
# <=50% missingness criterion.
#
#
# PURPOSE
# -------
# Determine whether adding the structurally restricted parent
# questionnaire predictor PA006Q09TA:
#
#   1. materially improves prediction
#   2. materially changes the 46-variable architecture
#   3. becomes an important predictor itself
#   4. changes BELONG's dominance
#
#
# Expanded46 models are REUSED from Phase 3E.
# Only Full47 models are fitted.
#
# 12 systems
# 59 valid Country × Outcome combinations
# 10 matched plausible-value replicates each
# ============================================================


from pathlib import Path
import pickle
import gc
import time

import numpy as np
import pandas as pd

from scipy.stats import spearmanr


# ============================================================
# 1. Paths
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "expanded_predictor_analysis"
)


EXPANDED46_CHECKPOINT_DIR = (
    OUT_DIR
    / "core17_vs_expanded46_checkpoints"
)


FULL47_CHECKPOINT_DIR = (
    OUT_DIR
    / "expanded46_vs_full47_parent_checkpoints"
)


FOLD_DIR = (
    OUT_DIR
    / "frozen_primary53_folds"
)


FULL47_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. Verify required Phase 3E functions exist
# ============================================================

required_functions = [

    "numeric_frame",
    "fit_expanded46_oof",
    "canonical_feature",
    "cosine_similarity",
]


for name in required_functions:

    if name not in globals():

        raise RuntimeError(
            f"{name} not found. "
            "Run the Phase 3E cell first."
        )


# ============================================================
# 3. Recover frozen Parent12 cohort
# ============================================================

cohort_path = (
    OUT_DIR
    / "frozen_expanded_system_cohorts.csv"
)


cohorts = pd.read_csv(
    cohort_path
)


PARENT12 = sorted(

    cohorts.loc[
        cohorts[
            "FullParent12"
        ]
        ==
        True,
        "Country"
    ]
    .unique()
)


assert len(
    PARENT12
) == 12


print(
    "Parent-module systems:",
    len(
        PARENT12
    )
)


print(
    PARENT12
)


# ============================================================
# 4. Exact valid country × outcome universe
# ============================================================

primary_models = pd.read_csv(
    OUT_DIR
    / "frozen_primary53_country_outcome_models.csv"
)


parent_models = (
    primary_models[
        primary_models[
            "Country"
        ]
        .isin(
            PARENT12
        )
    ]
    .sort_values(
        [
            "Country",
            "Outcome"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nParent-module country × outcome models:",
    len(
        parent_models
    )
)


assert len(
    parent_models
) == 59


# ============================================================
# 5. Predictor sets
# ============================================================

EXPANDED43 = [
    predictor
    for predictor
    in EXPANDED_NONACHIEVEMENT
    if predictor
    !=
    "PA006Q09TA"
]


SHARED46_CONCEPTUAL = (
    EXPANDED43
    +
    [
        "PV_READ",
        "PV_MATH",
        "PV_SCIE",
    ]
)


assert len(
    EXPANDED43
) == 43


assert len(
    SHARED46_CONCEPTUAL
) == 46


# ============================================================
# 6. Compare Expanded46 and Full47 attribution
# ============================================================

def compare_expanded46_full47(
    expanded46_attr,
    full47_attr
):

    expanded = (
        expanded46_attr
        .set_index(
            "CanonicalFeature"
        )
        .copy()
    )


    full = (
        full47_attr
        .set_index(
            "CanonicalFeature"
        )
        .copy()
    )


    if set(
        SHARED46_CONCEPTUAL
    ) != set(
        expanded.index
    ):

        raise RuntimeError(
            "Expanded46 conceptual feature set mismatch."
        )


    if (
        "PA006Q09TA"
        not in full.index
    ):

        raise RuntimeError(
            "PA006Q09TA missing from Full47 attribution."
        )


    # --------------------------------------------------------
    # Shared46 attribution mass within Full47
    # --------------------------------------------------------

    shared_mass = float(

        full.loc[
            SHARED46_CONCEPTUAL,
            "ImportanceShare"
        ]
        .sum()
    )


    # --------------------------------------------------------
    # Renormalize Full47's shared 46 after parent removal
    # --------------------------------------------------------

    full_shared = (

        full.loc[
            SHARED46_CONCEPTUAL,
            "ImportanceShare"
        ]

        /

        shared_mass
    )


    expanded_share = (

        expanded.loc[
            SHARED46_CONCEPTUAL,
            "ImportanceShare"
        ]
    )


    cosine = cosine_similarity(
        expanded_share,
        full_shared
    )


    rank_expanded = (
        expanded_share
        .rank(
            ascending=False,
            method="average"
        )
    )


    rank_full = (
        full_shared
        .rank(
            ascending=False,
            method="average"
        )
    )


    rho = spearmanr(
        rank_expanded,
        rank_full
    ).statistic


    parent_share = float(
        full.loc[
            "PA006Q09TA",
            "ImportanceShare"
        ]
    )


    parent_rank = int(
        full.loc[
            "PA006Q09TA",
            "Rank"
        ]
    )


    belong_expanded_rank = int(
        expanded.loc[
            "BELONG",
            "Rank"
        ]
    )


    belong_full_rank = int(
        full.loc[
            "BELONG",
            "Rank"
        ]
    )


    achievement_expanded = float(

        expanded.loc[
            [
                "PV_READ",
                "PV_MATH",
                "PV_SCIE"
            ],
            "ImportanceShare"
        ]
        .sum()
    )


    achievement_full = float(

        full.loc[
            [
                "PV_READ",
                "PV_MATH",
                "PV_SCIE"
            ],
            "ImportanceShare"
        ]
        .sum()
    )


    return {

        "Shared46AttributionMass":
            shared_mass,

        "Shared46RenormCosine":
            cosine,

        "Shared46RankSpearman":
            float(
                rho
            ),

        "ParentShare":
            parent_share,

        "ParentRank":
            parent_rank,

        "BELONG_Expanded46Rank":
            belong_expanded_rank,

        "BELONG_Full47Rank":
            belong_full_rank,

        "BELONG_RankChange":
            (
                belong_full_rank
                -
                belong_expanded_rank
            ),

        "Expanded46AchievementShare":
            achievement_expanded,

        "Full47AchievementShare":
            achievement_full,
    }


# ============================================================
# 7. Main Full47 fitting loop
# ============================================================

start_time = time.time()


for pair_index, pair in (
    parent_models.iterrows()
):

    country = pair[
        "Country"
    ]


    outcome = pair[
        "Outcome"
    ]


    full_checkpoint_path = (
        FULL47_CHECKPOINT_DIR
        /
        f"{country}__{outcome}.pkl"
    )


    if full_checkpoint_path.exists():

        print(
            f"[{pair_index + 1:02d}/{len(parent_models)}] "
            f"{country} {outcome} — already complete"
        )


        continue


    print(
        "\n"
        +
        "=" * 100
    )


    print(
        f"[{pair_index + 1:02d}/{len(parent_models)}] "
        f"{country} | {outcome}"
    )


    pair_start = time.time()


    # --------------------------------------------------------
    # Retrieve completed Expanded46 checkpoint
    # --------------------------------------------------------

    expanded_checkpoint_path = (
        EXPANDED46_CHECKPOINT_DIR
        /
        f"{country}__{outcome}.pkl"
    )


    if not expanded_checkpoint_path.exists():

        raise FileNotFoundError(
            f"Missing Expanded46 checkpoint: "
            f"{country} {outcome}"
        )


    with open(
        expanded_checkpoint_path,
        "rb"
    ) as file:

        expanded_results = (
            pickle.load(
                file
            )
        )


    if len(
        expanded_results
    ) != 10:

        raise RuntimeError(
            "Expanded46 checkpoint does not contain 10 PVs."
        )


    expanded_by_pv = {
        int(
            row[
                "PV"
            ]
        ):
        row
        for row
        in expanded_results
    }


    # --------------------------------------------------------
    # Load processed system data
    # --------------------------------------------------------

    df = pd.read_pickle(

        PROCESSED_DIR
        /
        f"{country}_analysis.pkl"
    )


    y_full = pd.to_numeric(
        df[
            outcome
        ],
        errors="coerce"
    )


    weight_full = pd.to_numeric(
        df[
            "W_FSTUWT"
        ],
        errors="coerce"
    )


    valid_mask = (

        np.isfinite(
            y_full
        )

        &

        np.isfinite(
            weight_full
        )

        &

        (
            weight_full
            >
            0
        )

        &

        df[
            "CNTSCHID"
        ]
        .notna()
    )


    selected = (
        df.loc[
            valid_mask
        ]
        .copy()
    )


    y = (
        pd.to_numeric(
            selected[
                outcome
            ],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    weights = (
        pd.to_numeric(
            selected[
                "W_FSTUWT"
            ],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    student_ids = (
        selected[
            "CNTSTUID"
        ]
        .astype(
            str
        )
        .to_numpy()
    )


    # --------------------------------------------------------
    # Exact frozen folds from Phase 3D / 3E
    # --------------------------------------------------------

    fold_path = (

        FOLD_DIR
        /
        f"{country}__{outcome}__folds.pkl"
    )


    fold_table = pd.read_pickle(
        fold_path
    )


    fold_map = (

        fold_table
        .set_index(
            "CNTSTUID"
        )[
            "Fold"
        ]
    )


    fold_ids = (
        pd.Series(
            student_ids
        )
        .map(
            fold_map
        )
        .to_numpy(
            dtype=int
        )
    )


    if (
        pd.isna(
            fold_ids
        )
        .any()
    ):

        raise RuntimeError(
            "Frozen fold mapping failed."
        )


    # --------------------------------------------------------
    # Base 44:
    #
    # Expanded43 + parent questionnaire item
    # --------------------------------------------------------

    BASE44 = (
        EXPANDED43
        +
        [
            "PA006Q09TA"
        ]
    )


    assert len(
        BASE44
    ) == 44


    X_base44 = numeric_frame(
        selected,
        BASE44
    )


    pair_results = []


    # ========================================================
    # 10 matched PV replicates
    # ========================================================

    for pv in range(
        1,
        11
    ):

        print(
            f"   PV {pv}/10",
            end=""
        )


        read_col = (
            f"PV{pv}READ"
        )


        math_col = (
            f"PV{pv}MATH"
        )


        scie_col = (
            f"PV{pv}SCIE"
        )


        full47_features = (
            BASE44
            +
            [
                read_col,
                math_col,
                scie_col,
            ]
        )


        assert len(
            full47_features
        ) == 47


        X = X_base44.copy()


        X[
            read_col
        ] = pd.to_numeric(
            selected[
                read_col
            ],
            errors="coerce"
        )


        X[
            math_col
        ] = pd.to_numeric(
            selected[
                math_col
            ],
            errors="coerce"
        )


        X[
            scie_col
        ] = pd.to_numeric(
            selected[
                scie_col
            ],
            errors="coerce"
        )


        # ====================================================
        # Fit ONLY Full47
        # ====================================================

        full_perf, full_attr = (
            fit_expanded46_oof(

                X=X,

                y=y,

                weights=weights,

                fold_ids=fold_ids,

                outcome=outcome,

                features=full47_features,
            )
        )


        full_attr[
            "CanonicalFeature"
        ] = [
            canonical_feature(
                raw_feature=feature
            )
            for feature
            in full_attr[
                "Feature"
            ]
        ]


        if (
            full_attr[
                "CanonicalFeature"
            ]
            .nunique()
            !=
            47
        ):

            raise RuntimeError(
                "Full47 did not map to 47 unique "
                "conceptual predictors."
            )


        # ====================================================
        # Existing Expanded46 result
        # ====================================================

        expanded_result = (
            expanded_by_pv[
                pv
            ]
        )


        expanded_attr = (
            expanded_result[
                "ExpandedAttribution"
            ]
            .copy()
        )


        if (
            "CanonicalFeature"
            not in expanded_attr.columns
        ):

            expanded_attr[
                "CanonicalFeature"
            ] = [
                canonical_feature(
                    raw_feature=feature
                )
                for feature
                in expanded_attr[
                    "Feature"
                ]
            ]


        # ====================================================
        # Performance delta
        # ====================================================

        if outcome == "BEINGBULLIED":

            expanded_metric = float(
                expanded_result[
                    "ExpandedR2"
                ]
            )


            full_metric = float(
                full_perf[
                    "R2"
                ]
            )


            primary_metric = (
                "R2"
            )


        else:

            expanded_metric = float(
                expanded_result[
                    "ExpandedROC_AUC"
                ]
            )


            full_metric = float(
                full_perf[
                    "ROC_AUC"
                ]
            )


            primary_metric = (
                "ROC_AUC"
            )


        # ====================================================
        # Architecture comparison
        # ====================================================

        architecture = (
            compare_expanded46_full47(

                expanded46_attr=
                    expanded_attr,

                full47_attr=
                    full_attr,
            )
        )


        # ====================================================
        # Store full attribution
        # ====================================================

        full_attr = (
            full_attr.copy()
        )


        full_attr.insert(
            0,
            "PV",
            pv
        )


        full_attr.insert(
            0,
            "Outcome",
            outcome
        )


        full_attr.insert(
            0,
            "Country",
            country
        )


        pair_results.append(
            {

                "PV":
                    pv,

                "Country":
                    country,

                "Outcome":
                    outcome,

                "N":
                    len(
                        y
                    ),

                "PrimaryMetric":
                    primary_metric,

                "Expanded46Metric":
                    expanded_metric,

                "Full47Metric":
                    full_metric,

                "DeltaFull47":
                    (
                        full_metric
                        -
                        expanded_metric
                    ),

                "Full47R2":
                    full_perf[
                        "R2"
                    ],

                "Full47ROC_AUC":
                    full_perf[
                        "ROC_AUC"
                    ],

                "Full47PR_AUC":
                    full_perf[
                        "PR_AUC"
                    ],

                **architecture,

                "Full47Attribution":
                    full_attr,
            }
        )


        print(
            " | Δ=",
            round(
                full_metric
                -
                expanded_metric,
                4
            ),
            "| parent rank=",
            architecture[
                "ParentRank"
            ],
            "| parent share=",
            round(
                architecture[
                    "ParentShare"
                ],
                4
            )
        )


        del (
            X,
            full_attr
        )


        gc.collect()


    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    with open(
        full_checkpoint_path,
        "wb"
    ) as file:

        pickle.dump(
            pair_results,
            file
        )


    print(
        "   Pair complete in",
        round(
            (
                time.time()
                -
                pair_start
            )
            /
            60,
            2
        ),
        "min"
    )


    del (
        df,
        selected,
        X_base44,
        expanded_results,
        pair_results
    )


    gc.collect()


# ============================================================
# 8. Reassemble all checkpoints
# ============================================================

comparison_rows = []

full47_attribution = []


for _, pair in (
    parent_models.iterrows()
):

    country = pair[
        "Country"
    ]


    outcome = pair[
        "Outcome"
    ]


    checkpoint_path = (

        FULL47_CHECKPOINT_DIR
        /
        f"{country}__{outcome}.pkl"
    )


    with open(
        checkpoint_path,
        "rb"
    ) as file:

        rows = pickle.load(
            file
        )


    if len(
        rows
    ) != 10:

        raise RuntimeError(
            "Incomplete Full47 checkpoint."
        )


    for row in rows:

        full47_attribution.append(
            row[
                "Full47Attribution"
            ]
        )


        comparison_rows.append(
            {
                key:
                    value
                for key, value
                in row.items()
                if key
                !=
                "Full47Attribution"
            }
        )


comparison_df = pd.DataFrame(
    comparison_rows
)


full47_attr_df = pd.concat(
    full47_attribution,
    ignore_index=True
)


assert len(
    comparison_df
) == (
    59
    *
    10
)


# ============================================================
# 9. SHAP QA
# ============================================================

share_sum = (

    full47_attr_df
    .groupby(
        [
            "PV",
            "Country",
            "Outcome"
        ]
    )[
        "ImportanceShare"
    ]
    .sum()
)


max_share_error = float(

    np.abs(
        share_sum
        -
        1
    )
    .max()
)


print(
    "\nMaximum Full47 SHAP share error:",
    max_share_error
)


# ============================================================
# 10. Save PV-level output
# ============================================================

comparison_df.to_csv(

    OUT_DIR
    / "expanded46_vs_full47_parent_pv_comparison.csv",

    index=False
)


full47_attr_df.to_csv(

    OUT_DIR
    / "full47_parent_pv_oof_shap_rankings.csv",

    index=False
)


# ============================================================
# 11. Average across 10 PVs within each system
# ============================================================

system_comparison = (

    comparison_df
    .groupby(
        [
            "Country",
            "Outcome",
            "PrimaryMetric"
        ],
        as_index=False
    )
    .agg(

        Expanded46Metric=(
            "Expanded46Metric",
            "mean"
        ),

        Full47Metric=(
            "Full47Metric",
            "mean"
        ),

        DeltaFull47=(
            "DeltaFull47",
            "mean"
        ),

        Shared46AttributionMass=(
            "Shared46AttributionMass",
            "mean"
        ),

        Shared46RenormCosine=(
            "Shared46RenormCosine",
            "mean"
        ),

        Shared46RankSpearman=(
            "Shared46RankSpearman",
            "mean"
        ),

        ParentShare=(
            "ParentShare",
            "mean"
        ),

        ParentMeanRank=(
            "ParentRank",
            "mean"
        ),

        ParentBestRank=(
            "ParentRank",
            "min"
        ),

        BELONG_Expanded46Rank=(
            "BELONG_Expanded46Rank",
            "mean"
        ),

        BELONG_Full47Rank=(
            "BELONG_Full47Rank",
            "mean"
        ),

        Expanded46AchievementShare=(
            "Expanded46AchievementShare",
            "mean"
        ),

        Full47AchievementShare=(
            "Full47AchievementShare",
            "mean"
        ),
    )
)


system_comparison.to_csv(

    OUT_DIR
    / "expanded46_vs_full47_parent_system_comparison.csv",

    index=False
)


# ============================================================
# 12. Full47 conceptual attribution averaged over PVs
# ============================================================

full47_conceptual = (

    full47_attr_df
    .groupby(
        [
            "Country",
            "Outcome",
            "CanonicalFeature"
        ],
        as_index=False
    )
    .agg(

        MeanImportanceShare=(
            "ImportanceShare",
            "mean"
        ),

        MeanRankAcrossPVs=(
            "Rank",
            "mean"
        ),
    )
)


full47_conceptual[
    "ConceptualRank"
] = (

    full47_conceptual
    .groupby(
        [
            "Country",
            "Outcome"
        ]
    )[
        "MeanImportanceShare"
    ]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(
        int
    )
)


full47_conceptual.to_csv(

    OUT_DIR
    / "full47_parent_system_conceptual_summary.csv",

    index=False
)


# ============================================================
# 13. Outcome-level performance sensitivity
# ============================================================

performance_summary = (

    system_comparison
    .groupby(
        "Outcome",
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        Mean_Expanded46=(
            "Expanded46Metric",
            "mean"
        ),

        Mean_Full47=(
            "Full47Metric",
            "mean"
        ),

        Mean_Delta=(
            "DeltaFull47",
            "mean"
        ),

        Median_Delta=(
            "DeltaFull47",
            "median"
        ),

        Min_Delta=(
            "DeltaFull47",
            "min"
        ),

        Max_Delta=(
            "DeltaFull47",
            "max"
        ),

        N_Full47Better=(
            "DeltaFull47",
            lambda x:
                int(
                    (
                        x
                        >
                        0
                    )
                    .sum()
                )
        ),
    )
)


# ============================================================
# 14. Parent predictor importance by outcome
# ============================================================

parent_importance = (

    full47_conceptual[
        full47_conceptual[
            "CanonicalFeature"
        ]
        ==
        "PA006Q09TA"
    ]
    .groupby(
        "Outcome",
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        MeanShare=(
            "MeanImportanceShare",
            "mean"
        ),

        MedianShare=(
            "MeanImportanceShare",
            "median"
        ),

        MeanRank=(
            "ConceptualRank",
            "mean"
        ),

        MedianRank=(
            "ConceptualRank",
            "median"
        ),

        Rank1Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        ==
                        1
                    )
                    .sum()
                )
        ),

        Top5Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        <=
                        5
                    )
                    .sum()
                )
        ),

        Top10Count=(
            "ConceptualRank",
            lambda x:
                int(
                    (
                        x
                        <=
                        10
                    )
                    .sum()
                )
        ),
    )
)


# ============================================================
# 15. Architecture stability
# ============================================================

architecture_summary = (

    system_comparison
    .groupby(
        "Outcome",
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "count"
        ),

        Mean_Shared46Mass=(
            "Shared46AttributionMass",
            "mean"
        ),

        Mean_RenormCosine=(
            "Shared46RenormCosine",
            "mean"
        ),

        Mean_RankSpearman=(
            "Shared46RankSpearman",
            "mean"
        ),

        Mean_BELONG_Expanded46Rank=(
            "BELONG_Expanded46Rank",
            "mean"
        ),

        Mean_BELONG_Full47Rank=(
            "BELONG_Full47Rank",
            "mean"
        ),

        Mean_Expanded46AchievementShare=(
            "Expanded46AchievementShare",
            "mean"
        ),

        Mean_Full47AchievementShare=(
            "Full47AchievementShare",
            "mean"
        ),
    )
)


# ============================================================
# 16. Save summaries
# ============================================================

performance_summary.to_csv(

    OUT_DIR
    / "expanded46_vs_full47_parent_performance_summary.csv",

    index=False
)


parent_importance.to_csv(

    OUT_DIR
    / "pa006_parent_predictor_importance_summary.csv",

    index=False
)


architecture_summary.to_csv(

    OUT_DIR
    / "expanded46_vs_full47_parent_architecture_summary.csv",

    index=False
)


# ============================================================
# 17. Print
# ============================================================

print("\n")
print("=" * 125)
print("EXPANDED46 vs FULL47 — PARENT-MODULE PERFORMANCE")
print("=" * 125)


print(
    performance_summary
    .round(
        5
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 125)
print("PA006Q09TA IMPORTANCE INSIDE FULL47")
print("=" * 125)


print(
    parent_importance
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 125)
print("EXPANDED46 ARCHITECTURE AFTER ADDING PA006Q09TA")
print("=" * 125)


print(
    architecture_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 125)
print("PHASE 3F LITERAL FULL47 SENSITIVITY COMPLETE")
print("=" * 125)


print(
    "PV-level comparisons:",
    len(
        comparison_df
    )
)


print(
    "System × outcome comparisons:",
    len(
        system_comparison
    )
)


print(
    "Runtime:",
    round(
        (
            time.time()
            -
            start_time
        )
        /
        60,
        2
    ),
    "minutes"
)


print(
    "\nAfter this result, Phase 3 can be frozen."
)

Parent-module systems: 12
['BRA', 'CHL', 'GEO', 'HKG', 'HRV', 'IRL', 'KOR', 'LUX', 'MAC', 'MEX', 'MLT', 'PRT']

Parent-module country × outcome models: 59

[01/59] BRA | BEINGBULLIED
   PV 1/10 | Δ= 0.0051 | parent rank= 39 | parent share= 0.0078
   PV 2/10 | Δ= -0.0006 | parent rank= 39 | parent share= 0.008
   PV 3/10 | Δ= 0.0026 | parent rank= 39 | parent share= 0.008
   PV 4/10 | Δ= 0.0039 | parent rank= 40 | parent share= 0.007
   PV 5/10 | Δ= 0.0006 | parent rank= 40 | parent share= 0.0076
   PV 6/10 | Δ= 0.0043 | parent rank= 40 | parent share= 0.0067
   PV 7/10 | Δ= 0.0007 | parent rank= 40 | parent share= 0.007
   PV 8/10 | Δ= 0.001 | parent rank= 41 | parent share= 0.0065
   PV 9/10 | Δ= -0.0067 | parent rank= 42 | parent share= 0.006
   PV 10/10 | Δ= -0.0011 | parent rank= 40 | parent share= 0.0077
   Pair complete in 0.54 min

[02/59] BRA | BULLY_PHYSICAL
   PV 1/10 | Δ= -0.0042 | parent rank= 46 | parent share= 0.0043
   PV 2/10 | Δ= -0.0029 | parent rank= 45 | parent shar